In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/RATE_FP_Project")
DATA_DIR = BASE_DIR / "data"

projects = {
    "jdt": DATA_DIR / "jdt",
    "pde": DATA_DIR / "pde",
    "equinox": DATA_DIR / "equinox",
    "lucene": DATA_DIR / "lucene",
    "mylyn": DATA_DIR / "mylyn"
}

for name, path in projects.items():
    print("\n" + "="*80)
    print("Project:", name)
    print("Folder exists:", path.exists())

    if path.exists():
        for f in path.iterdir():
            print(" -", f.name)

Mounted at /content/drive

Project: jdt
Folder exists: True
 - change-metrics.csv
 - single-version-ck-oo.csv
 - biweekly-ck-values.zip
 - biweekly-oo-values.zip
 - biweekly-ck-values
 - biweekly-oo-values

Project: pde
Folder exists: True
 - biweekly-ck-values.zip
 - biweekly-oo-values.zip
 - change-metrics.csv
 - single-version-ck-oo.csv
 - biweekly-ck-values
 - biweekly-oo-values

Project: equinox
Folder exists: True
 - biweekly-ck-values.zip
 - biweekly-oo-values.zip
 - single-version-ck-oo.csv
 - change-metrics.csv
 - biweekly-ck-values
 - biweekly-oo-values

Project: lucene
Folder exists: True
 - biweekly-ck-values.zip
 - biweekly-oo-values.zip
 - change-metrics.csv
 - single-version-ck-oo.csv
 - biweekly-ck-values
 - biweekly-oo-values

Project: mylyn
Folder exists: True
 - biweekly-ck-values.zip
 - biweekly-oo-values.zip
 - change-metrics.csv
 - single-version-ck-oo.csv
 - biweekly-ck-values
 - biweekly-oo-values


In [ ]:
import zipfile

for project_name, project_path in projects.items():
    print("\n" + "="*80)
    print("Processing:", project_name)

    for file in project_path.iterdir():
        if file.suffix.lower() == ".zip":
            extract_dir = project_path / file.stem
            extract_dir.mkdir(exist_ok=True)

            # Skip if already extracted
            if any(extract_dir.iterdir()):
                print(f"Already extracted: {file.name}")
                continue

            with zipfile.ZipFile(file, "r") as zip_ref:
                zip_ref.extractall(extract_dir)

            print(f"Extracted: {file.name} -> {extract_dir}")


Processing: jdt
Already extracted: biweekly-ck-values.zip
Already extracted: biweekly-oo-values.zip

Processing: pde
Already extracted: biweekly-ck-values.zip
Already extracted: biweekly-oo-values.zip

Processing: equinox
Already extracted: biweekly-ck-values.zip
Already extracted: biweekly-oo-values.zip

Processing: lucene
Already extracted: biweekly-ck-values.zip
Already extracted: biweekly-oo-values.zip

Processing: mylyn
Already extracted: biweekly-ck-values.zip
Already extracted: biweekly-oo-values.zip


In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib openpyxl xlrd torch torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.9 MB/s eta 0:00:00


In [ ]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Torch version: 2.10.0+cu128
CUDA available: True
Device: cuda


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LinearRegression

PROCESSED_DIR = BASE_DIR / "processed_all"
RESULTS_DIR = BASE_DIR / "results"
FIG_DIR = BASE_DIR / "figures_all"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)


def clean_column_names(df):
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.replace(":", "", regex=False)
        .str.replace(" ", "_", regex=False)
    )
    return df


def load_semicolon_csv(path):
    df = pd.read_csv(path, sep=";", engine="python")
    df = df.dropna(axis=1, how="all")
    df = clean_column_names(df)

    # Remove empty unnamed columns
    cols_to_drop = [col for col in df.columns if str(col).strip() == "" or str(col).startswith("_")]
    df = df.drop(columns=cols_to_drop, errors="ignore")

    df["classname"] = df["classname"].astype(str).str.strip()

    for col in df.columns:
        if col != "classname":
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.fillna(0)
    return df


def load_biweekly_metric(path, metric_name):
    df = pd.read_csv(path, sep=";", header=None, engine="python")
    df = df.dropna(axis=1, how="all")

    n_cols = df.shape[1]
    df.columns = ["classname"] + [f"{metric_name}_t{i}" for i in range(1, n_cols)]

    df["classname"] = df["classname"].astype(str).str.strip()

    for col in df.columns[1:]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def load_all_biweekly_metrics(folder_path, prefix_text):
    temporal_tables = []

    for file in sorted(folder_path.glob("*.csv")):
        metric_name = file.stem.replace(prefix_text, "")
        temp_df = load_biweekly_metric(file, metric_name)
        temporal_tables.append(temp_df)

    if len(temporal_tables) == 0:
        raise FileNotFoundError(f"No CSV files found inside {folder_path}")

    temporal_df = temporal_tables[0]

    for df in temporal_tables[1:]:
        temporal_df = pd.merge(temporal_df, df, on="classname", how="outer")

    return temporal_df


def compute_slope(values):
    values = np.array(values, dtype=float)
    valid_mask = ~np.isnan(values)

    if valid_mask.sum() < 2:
        return 0.0

    y_vals = values[valid_mask]
    x_vals = np.arange(len(values))[valid_mask].reshape(-1, 1)

    model = LinearRegression()
    model.fit(x_vals, y_vals)

    return float(model.coef_[0])


def compute_temporal_features_for_metric(df, metric_name):
    cols = [col for col in df.columns if col.startswith(metric_name + "_t")]
    metric_values = df[cols].values.astype(float)

    features = pd.DataFrame()
    features["classname"] = df["classname"]

    features[f"{metric_name}_temp_mean"] = np.nanmean(metric_values, axis=1)
    features[f"{metric_name}_temp_std"] = np.nanstd(metric_values, axis=1)
    features[f"{metric_name}_temp_min"] = np.nanmin(metric_values, axis=1)
    features[f"{metric_name}_temp_max"] = np.nanmax(metric_values, axis=1)

    first_vals = []
    last_vals = []
    change_counts = []

    for row in metric_values:
        valid = row[~np.isnan(row)]

        if len(valid) == 0:
            first_vals.append(0)
            last_vals.append(0)
            change_counts.append(0)
        else:
            first_vals.append(valid[0])
            last_vals.append(valid[-1])
            change_counts.append(np.sum(np.diff(valid) != 0) if len(valid) > 1 else 0)

    features[f"{metric_name}_temp_first"] = first_vals
    features[f"{metric_name}_temp_last"] = last_vals
    features[f"{metric_name}_temp_growth"] = features[f"{metric_name}_temp_last"] - features[f"{metric_name}_temp_first"]
    features[f"{metric_name}_temp_change_count"] = change_counts
    features[f"{metric_name}_temp_slope"] = [compute_slope(row) for row in metric_values]

    return features


def extract_temporal_features(temporal_df):
    temporal_clean = temporal_df.copy()
    metric_cols = [col for col in temporal_clean.columns if col != "classname"]

    # -1 means class unavailable or metric unavailable
    temporal_clean[metric_cols] = temporal_clean[metric_cols].replace(-1, np.nan)

    metric_names = sorted(set([col.split("_t")[0] for col in temporal_clean.columns if "_t" in col]))

    temporal_feature_tables = []

    for metric in metric_names:
        temp_feat = compute_temporal_features_for_metric(temporal_clean, metric)
        temporal_feature_tables.append(temp_feat)

    temporal_features_df = temporal_feature_tables[0]

    for df in temporal_feature_tables[1:]:
        temporal_features_df = pd.merge(temporal_features_df, df, on="classname", how="outer")

    temporal_features_df = temporal_features_df.fillna(0)
    return temporal_features_df, metric_names

In [ ]:
def process_project_dataset(project_name, project_path):
    print("\n" + "="*100)
    print(f"Processing project: {project_name}")

    change_path = project_path / "change-metrics.csv"
    ckoo_path = project_path / "single-version-ck-oo.csv"

    change_df = load_semicolon_csv(change_path)
    ckoo_df = load_semicolon_csv(ckoo_path)

    target_cols = ["bugs", "nonTrivialBugs", "majorBugs", "criticalBugs", "highPriorityBugs"]

    ckoo_features = ckoo_df.drop(columns=[c for c in target_cols if c in ckoo_df.columns], errors="ignore")

    baseline_df = pd.merge(
        change_df,
        ckoo_features,
        on="classname",
        how="inner"
    )

    baseline_df["bugs"] = pd.to_numeric(baseline_df["bugs"], errors="coerce").fillna(0)
    baseline_df["is_defective"] = (baseline_df["bugs"] > 0).astype(int)

    ck_folders = list(project_path.glob("biweekly-ck-values/**"))
    oo_folders = list(project_path.glob("biweekly-oo-values/**"))

    ck_folder = None
    oo_folder = None

    for folder in ck_folders:
        if folder.is_dir() and len(list(folder.glob("*.csv"))) > 0:
            ck_folder = folder
            break

    for folder in oo_folders:
        if folder.is_dir() and len(list(folder.glob("*.csv"))) > 0:
            oo_folder = folder
            break

    if ck_folder is None:
        raise FileNotFoundError(f"Could not find extracted CK folder for {project_name}")
    if oo_folder is None:
        raise FileNotFoundError(f"Could not find extracted OO folder for {project_name}")

    ck_temporal_df = load_all_biweekly_metrics(ck_folder, "eclipse-class-")
    oo_temporal_df = load_all_biweekly_metrics(oo_folder, "eclipse-class-")

    temporal_df = pd.merge(ck_temporal_df, oo_temporal_df, on="classname", how="outer")
    temporal_features_df, metric_names = extract_temporal_features(temporal_df)

    rate_fp_df = pd.merge(
        baseline_df,
        temporal_features_df,
        on="classname",
        how="inner"
    )

    for col in rate_fp_df.columns:
        if col != "classname":
            rate_fp_df[col] = pd.to_numeric(rate_fp_df[col], errors="coerce")

    rate_fp_df = rate_fp_df.fillna(0)
    rate_fp_df["is_defective"] = (rate_fp_df["bugs"] > 0).astype(int)

    project_out_dir = PROCESSED_DIR / project_name
    project_out_dir.mkdir(parents=True, exist_ok=True)

    baseline_df.to_csv(project_out_dir / f"{project_name}_baseline_static.csv", index=False)
    temporal_features_df.to_csv(project_out_dir / f"{project_name}_temporal_features.csv", index=False)
    rate_fp_df.to_csv(project_out_dir / f"{project_name}_rate_fp_static_temporal.csv", index=False)

    print("Change shape:", change_df.shape)
    print("CKOO shape:", ckoo_df.shape)
    print("Baseline merged shape:", baseline_df.shape)
    print("Temporal raw shape:", temporal_df.shape)
    print("Temporal extracted shape:", temporal_features_df.shape)
    print("RATE-FP dataset shape:", rate_fp_df.shape)
    print("Class distribution:")
    print(rate_fp_df["is_defective"].value_counts())

    return rate_fp_df

In [ ]:
all_project_data = {}

for project_name, project_path in projects.items():
    try:
        df = process_project_dataset(project_name, project_path)
        all_project_data[project_name] = df
    except Exception as e:
        print("\nERROR in project:", project_name)
        print(e)


Processing project: jdt
Change shape: (997, 21)
CKOO shape: (997, 23)
Baseline merged shape: (997, 39)
Temporal raw shape: (1041, 1531)
Temporal extracted shape: (1041, 154)
RATE-FP dataset shape: (997, 192)
Class distribution:
is_defective
0    791
1    206
Name: count, dtype: int64

Processing project: pde
Change shape: (1497, 21)
CKOO shape: (1497, 23)
Baseline merged shape: (1497, 39)
Temporal raw shape: (1924, 1650)
Temporal extracted shape: (1924, 154)
RATE-FP dataset shape: (1497, 192)
Class distribution:
is_defective
0    1288
1     209
Name: count, dtype: int64

Processing project: equinox
Change shape: (324, 21)
CKOO shape: (324, 23)
Baseline merged shape: (324, 39)
Temporal raw shape: (444, 1548)
Temporal extracted shape: (444, 154)
RATE-FP dataset shape: (324, 192)
Class distribution:
is_defective
0    195
1    129
Name: count, dtype: int64

Processing project: lucene
Change shape: (691, 21)
CKOO shape: (691, 23)
Baseline merged shape: (691, 39)
Temporal raw shape: (889, 1

In [ ]:
import time
import copy
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, BatchNorm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
class GraphSAGEDefectPredictor(nn.Module):
    def __init__(self, in_channels, hidden_channels=64, out_channels=2, dropout=0.40):
        super().__init__()

        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.bn1 = BatchNorm(hidden_channels)

        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.bn2 = BatchNorm(hidden_channels)

        self.classifier = nn.Linear(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        return self.classifier(x)


def evaluate_predictions(y_true, y_pred, y_prob):
    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = np.nan

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": auc
    }


def build_similarity_graph(X_scaled, k_neighbors=10):
    n_samples = X_scaled.shape[0]
    k_neighbors = min(k_neighbors, n_samples - 1)

    nbrs = NearestNeighbors(
        n_neighbors=k_neighbors + 1,
        metric="cosine"
    ).fit(X_scaled)

    distances, indices = nbrs.kneighbors(X_scaled)

    edge_list = []

    for i in range(indices.shape[0]):
        for j in indices[i][1:]:
            edge_list.append([i, j])
            edge_list.append([j, i])

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    return edge_index


def evaluate_graph_model(model, data_obj, mask):
    model.eval()

    with torch.no_grad():
        logits = model(data_obj.x, data_obj.edge_index)
        probs = F.softmax(logits, dim=1)[:, 1]
        preds = logits.argmax(dim=1)

        y_true = data_obj.y[mask].detach().cpu().numpy()
        y_pred = preds[mask].detach().cpu().numpy()
        y_prob = probs[mask].detach().cpu().numpy()

    return evaluate_predictions(y_true, y_pred, y_prob), y_prob, y_pred

In [ ]:
def train_project_models(project_name, rate_fp_df, random_state=42):
    print("\n" + "="*120)
    print(f"Training project: {project_name}")
    print("="*120)

    rate_fp_df = rate_fp_df.copy()

    for col in rate_fp_df.columns:
        if col != "classname":
            rate_fp_df[col] = pd.to_numeric(rate_fp_df[col], errors="coerce")

    rate_fp_df = rate_fp_df.fillna(0)
    rate_fp_df["is_defective"] = (rate_fp_df["bugs"] > 0).astype(int)

    target_cols = [
        "bugs", "nonTrivialBugs", "majorBugs",
        "criticalBugs", "highPriorityBugs", "is_defective"
    ]

    static_feature_cols = [
        col for col in rate_fp_df.columns
        if col not in ["classname"] + target_cols and "_temp_" not in col
    ]

    static_temporal_feature_cols = [
        col for col in rate_fp_df.columns
        if col not in ["classname"] + target_cols
    ]

    y = rate_fp_df["is_defective"].values
    indices_all = np.arange(len(rate_fp_df))

    # Skip only if extremely problematic
    if len(np.unique(y)) < 2:
        print(f"Skipping {project_name}: only one class present.")
        return None

    train_idx, temp_idx, y_train, y_temp = train_test_split(
        indices_all,
        y,
        test_size=0.30,
        random_state=random_state,
        stratify=y
    )

    val_idx, test_idx, y_val, y_test = train_test_split(
        temp_idx,
        y_temp,
        test_size=0.50,
        random_state=random_state,
        stratify=y_temp
    )

    results = []

    # =========================
    # 1. STATIC-ONLY BASELINES
    # =========================
    X_static = rate_fp_df[static_feature_cols].values

    scaler_static = StandardScaler()
    X_static_train = scaler_static.fit_transform(X_static[train_idx])
    X_static_test = scaler_static.transform(X_static[test_idx])

    static_models = {
        "GBR Static": GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=random_state),
        "RF Static": RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced", random_state=random_state),
        "LR Static": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=random_state),
        "SVC Static": SVC(kernel="rbf", C=10, probability=True, class_weight="balanced", random_state=random_state),
        "KNN Static": KNeighborsClassifier(n_neighbors=5),
        "DT Static": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=random_state)
    }

    for model_name, model in static_models.items():
        start = time.time()
        model.fit(X_static_train, y[train_idx])

        pred = model.predict(X_static_test)
        prob = model.predict_proba(X_static_test)[:, 1]

        metrics = evaluate_predictions(y[test_idx], pred, prob)
        metrics.update({
            "Project": project_name,
            "Model": model_name,
            "Feature_Set": "Static",
            "Training_Time_sec": time.time() - start
        })
        results.append(metrics)

    # =========================
    # 2. STATIC + TEMPORAL ML
    # =========================
    X_st = rate_fp_df[static_temporal_feature_cols].values

    scaler_st = StandardScaler()
    X_train = scaler_st.fit_transform(X_st[train_idx])
    X_val = scaler_st.transform(X_st[val_idx])
    X_test = scaler_st.transform(X_st[test_idx])

    st_models = {
        "RF ST": RandomForestClassifier(
            n_estimators=300, max_depth=10, class_weight="balanced", random_state=random_state
        ),
        "GBR ST": GradientBoostingClassifier(
            n_estimators=300, max_depth=3, random_state=random_state
        ),
        "SVC ST": SVC(
            kernel="rbf", C=10, probability=True, class_weight="balanced", random_state=random_state
        ),
        "LR ST": LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=random_state
        ),
        "KNN ST": KNeighborsClassifier(n_neighbors=5),
        "DT ST": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=random_state)
    }

    val_probs = {}
    test_probs = {}
    val_f1_scores = {}

    for model_name, model in st_models.items():
        start = time.time()
        model.fit(X_train, y[train_idx])

        val_pred = model.predict(X_val)
        val_prob = model.predict_proba(X_val)[:, 1]

        test_pred = model.predict(X_test)
        test_prob = model.predict_proba(X_test)[:, 1]

        val_probs[model_name] = val_prob
        test_probs[model_name] = test_prob
        val_f1_scores[model_name] = f1_score(y[val_idx], val_pred, zero_division=0)

        metrics = evaluate_predictions(y[test_idx], test_pred, test_prob)
        metrics.update({
            "Project": project_name,
            "Model": model_name,
            "Feature_Set": "Static + Temporal",
            "Training_Time_sec": time.time() - start
        })
        results.append(metrics)

    # =========================
    # 3. GRAPHSAGE ST
    # =========================
    X_graph_scaled = StandardScaler().fit_transform(X_st)

    x_tensor = torch.tensor(X_graph_scaled, dtype=torch.float)
    y_tensor = torch.tensor(y, dtype=torch.long)

    edge_index = build_similarity_graph(X_graph_scaled, k_neighbors=10)

    num_nodes = x_tensor.shape[0]

    train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(num_nodes, dtype=torch.bool)

    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True

    data_st = Data(
        x=x_tensor,
        edge_index=edge_index,
        y=y_tensor,
        train_mask=train_mask,
        val_mask=val_mask,
        test_mask=test_mask
    ).to(device)

    graph_model = GraphSAGEDefectPredictor(
        in_channels=data_st.x.shape[1],
        hidden_channels=64,
        out_channels=2,
        dropout=0.40
    ).to(device)

    class_counts = np.bincount(y[train_idx])
    class_weights = class_counts.sum() / (2.0 * class_counts)
    class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

    optimizer = torch.optim.AdamW(graph_model.parameters(), lr=0.003, weight_decay=1e-3)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_val_f1 = 0
    best_state = None
    patience = 50
    patience_counter = 0

    start_graph = time.time()

    for epoch in range(1, 401):
        graph_model.train()
        optimizer.zero_grad()

        logits = graph_model(data_st.x, data_st.edge_index)
        loss = criterion(logits[data_st.train_mask], data_st.y[data_st.train_mask])

        loss.backward()
        optimizer.step()

        val_metrics, _, _ = evaluate_graph_model(graph_model, data_st, data_st.val_mask)
        current_val_f1 = val_metrics["F1"]

        if current_val_f1 > best_val_f1:
            best_val_f1 = current_val_f1
            best_state = copy.deepcopy(graph_model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    if best_state is not None:
        graph_model.load_state_dict(best_state)

    graph_test_metrics, graph_test_prob, graph_test_pred = evaluate_graph_model(
        graph_model, data_st, data_st.test_mask
    )

    graph_val_metrics, graph_val_prob, graph_val_pred = evaluate_graph_model(
        graph_model, data_st, data_st.val_mask
    )

    graph_test_metrics.update({
        "Project": project_name,
        "Model": "RATE-FP GraphSAGE ST",
        "Feature_Set": "Graph Static + Temporal",
        "Training_Time_sec": time.time() - start_graph
    })
    results.append(graph_test_metrics)

    val_probs["GraphSAGE ST"] = graph_val_prob
    test_probs["GraphSAGE ST"] = graph_test_prob
    val_f1_scores["GraphSAGE ST"] = graph_val_metrics["F1"]

    # =========================
    # 4. ADAPTIVE HYBRID ENSEMBLE
    # =========================
    selected_for_hybrid = ["RF ST", "GBR ST", "SVC ST", "LR ST", "GraphSAGE ST"]

    raw_weights = np.array([val_f1_scores[m] for m in selected_for_hybrid], dtype=float) + 1e-6
    adaptive_weights = raw_weights / raw_weights.sum()

    final_val_prob = np.zeros_like(y[val_idx], dtype=float)
    final_test_prob = np.zeros_like(y[test_idx], dtype=float)

    for model_name, weight in zip(selected_for_hybrid, adaptive_weights):
        final_val_prob += weight * val_probs[model_name]
        final_test_prob += weight * test_probs[model_name]

    thresholds = np.arange(0.20, 0.81, 0.01)
    best_threshold = 0.50
    best_threshold_f1 = 0

    for threshold in thresholds:
        val_pred_threshold = (final_val_prob >= threshold).astype(int)
        val_f1 = f1_score(y[val_idx], val_pred_threshold, zero_division=0)

        if val_f1 > best_threshold_f1:
            best_threshold_f1 = val_f1
            best_threshold = threshold

    final_test_pred = (final_test_prob >= best_threshold).astype(int)

    hybrid_metrics = evaluate_predictions(y[test_idx], final_test_pred, final_test_prob)
    hybrid_metrics.update({
        "Project": project_name,
        "Model": "RATE-FP Adaptive Hybrid Ensemble",
        "Feature_Set": "Hybrid Static + Temporal + Graph",
        "Training_Time_sec": np.nan,
        "Threshold": best_threshold
    })
    results.append(hybrid_metrics)

    print(f"Completed {project_name}")
    print("Best Hybrid F1:", hybrid_metrics["F1"])
    print("Best Hybrid ROC-AUC:", hybrid_metrics["ROC_AUC"])
    print("Hybrid threshold:", best_threshold)
    print("Confusion matrix:")
    print(confusion_matrix(y[test_idx], final_test_pred))

    result_df = pd.DataFrame(results)

    project_result_path = RESULTS_DIR / f"{project_name}_all_model_results.csv"
    result_df.to_csv(project_result_path, index=False)

    return result_df

In [ ]:
all_project_results = []

for project_name, df in all_project_data.items():
    result_df = train_project_models(project_name, df, random_state=42)
    if result_df is not None:
        all_project_results.append(result_df)

all_results_df = pd.concat(all_project_results, ignore_index=True)

all_results_path = RESULTS_DIR / "ALL_PROJECTS_RATE_FP_RESULTS.csv"
all_results_df.to_csv(all_results_path, index=False)

print("Saved all project results to:")
print(all_results_path)

display(all_results_df.sort_values(["Project", "F1"], ascending=[True, False]))


Training project: jdt
Completed jdt
Best Hybrid F1: 0.6956521739130435
Best Hybrid ROC-AUC: 0.9151531580374085
Hybrid threshold: 0.3900000000000002
Confusion matrix:
[[105  14]
 [  7  24]]

Training project: pde
Completed pde
Best Hybrid F1: 0.45901639344262296
Best Hybrid ROC-AUC: 0.7736947123378783
Hybrid threshold: 0.38000000000000017
Confusion matrix:
[[178  16]
 [ 17  14]]

Training project: equinox
Completed equinox
Best Hybrid F1: 0.5142857142857142
Best Hybrid ROC-AUC: 0.743859649122807
Hybrid threshold: 0.5600000000000003
Confusion matrix:
[[23  7]
 [10  9]]

Training project: lucene
Completed lucene
Best Hybrid F1: 0.21621621621621623
Best Hybrid ROC-AUC: 0.6210526315789474
Hybrid threshold: 0.3000000000000001
Confusion matrix:
[[71 24]
 [ 5  4]]

Training project: mylyn
Completed mylyn
Best Hybrid F1: 0.4835164835164835
Best Hybrid ROC-AUC: 0.765209654098543
Hybrid threshold: 0.36000000000000015
Confusion matrix:
[[211  32]
 [ 15  22]]
Saved all project results to:
/content

,Accuracy,Precision,Recall,F1,ROC_AUC,Project,Model,Feature_Set,Training_Time_sec,Threshold
29,0.734694,0.636364,0.736842,0.682927,0.780702,equinox,RF Static,Static,0.371933,NaN
28,0.734694,0.650000,0.684211,0.666667,0.782456,equinox,GBR Static,Static,0.456622,NaN
31,0.714286,0.619048,0.684211,0.650000,0.728070,equinox,SVC Static,Static,0.032740,NaN
32,0.714286,0.631579,0.631579,0.631579,0.768421,equinox,KNN Static,Static,0.010926,NaN
36,0.673469,0.565217,0.684211,0.619048,0.770175,equinox,SVC ST,Static + Temporal,0.061856,NaN
...,...,...,...,...,...,...,...,...,...,...
22,0.817778,0.307692,0.258065,0.280702,0.730961,pde,SVC ST,Static + Temporal,0.655616,NaN
21,0.853333,0.428571,0.193548,0.266667,0.808447,pde,GBR ST,Static + Temporal,9.326127,NaN
24,0.866667,0.555556,0.161290,0.250000,0.650482,pde,KNN ST,Static + Temporal,0.038980,NaN
14,0.857778,0.454545,0.161290,0.238095,0.772531,pde,GBR Static,Static,1.336511,NaN


In [ ]:
summary_rows = []

for project in all_results_df["Project"].unique():
    project_df = all_results_df[all_results_df["Project"] == project].copy()

    baseline_df = project_df[
        ~project_df["Model"].str.contains("RATE-FP Adaptive Hybrid Ensemble", regex=False)
    ]

    best_baseline = baseline_df.sort_values("F1", ascending=False).iloc[0]

    rate_fp = project_df[
        project_df["Model"] == "RATE-FP Adaptive Hybrid Ensemble"
    ].iloc[0]

    summary_rows.append({
        "Project": project,
        "Best_Baseline_Model": best_baseline["Model"],
        "Best_Baseline_F1": best_baseline["F1"],
        "RATE_FP_F1": rate_fp["F1"],
        "F1_Improvement": rate_fp["F1"] - best_baseline["F1"],
        "Best_Baseline_ROC_AUC": best_baseline["ROC_AUC"],
        "RATE_FP_ROC_AUC": rate_fp["ROC_AUC"],
        "ROC_AUC_Improvement": rate_fp["ROC_AUC"] - best_baseline["ROC_AUC"],
        "RATE_FP_Accuracy": rate_fp["Accuracy"],
        "RATE_FP_Precision": rate_fp["Precision"],
        "RATE_FP_Recall": rate_fp["Recall"]
    })

summary_df = pd.DataFrame(summary_rows)

avg_row = {
    "Project": "Average",
    "Best_Baseline_Model": "-",
    "Best_Baseline_F1": summary_df["Best_Baseline_F1"].mean(),
    "RATE_FP_F1": summary_df["RATE_FP_F1"].mean(),
    "F1_Improvement": summary_df["F1_Improvement"].mean(),
    "Best_Baseline_ROC_AUC": summary_df["Best_Baseline_ROC_AUC"].mean(),
    "RATE_FP_ROC_AUC": summary_df["RATE_FP_ROC_AUC"].mean(),
    "ROC_AUC_Improvement": summary_df["ROC_AUC_Improvement"].mean(),
    "RATE_FP_Accuracy": summary_df["RATE_FP_Accuracy"].mean(),
    "RATE_FP_Precision": summary_df["RATE_FP_Precision"].mean(),
    "RATE_FP_Recall": summary_df["RATE_FP_Recall"].mean()
}

summary_df = pd.concat([summary_df, pd.DataFrame([avg_row])], ignore_index=True)

display(summary_df)

summary_path = RESULTS_DIR / "SUMMARY_BEST_BASELINE_VS_RATE_FP.csv"
summary_df.to_csv(summary_path, index=False)

print("Saved summary table to:")
print(summary_path)

,Project,Best_Baseline_Model,Best_Baseline_F1,RATE_FP_F1,F1_Improvement,Best_Baseline_ROC_AUC,RATE_FP_ROC_AUC,ROC_AUC_Improvement,RATE_FP_Accuracy,RATE_FP_Precision,RATE_FP_Recall
0,jdt,LR ST,0.695652,0.695652,0.000000,0.887503,0.915153,0.027650,0.860000,0.631579,0.774194
1,pde,RATE-FP GraphSAGE ST,0.461538,0.459016,-0.002522,0.822248,0.773695,-0.048553,0.853333,0.466667,0.451613
2,equinox,RF Static,0.682927,0.514286,-0.168641,0.780702,0.743860,-0.036842,0.653061,0.562500,0.473684
3,lucene,RATE-FP GraphSAGE ST,0.413793,0.216216,-0.197577,0.750877,0.621053,-0.129825,0.721154,0.142857,0.444444
4,mylyn,RF Static,0.583333,0.483516,-0.099817,0.835836,0.765210,-0.070626,0.832143,0.407407,0.594595
5,Average,-,0.567449,0.473737,-0.093711,0.815433,0.763794,-0.051639,0.783938,0.442202,0.547706


Saved summary table to:
/content/drive/MyDrive/RATE_FP_Project/results/SUMMARY_BEST_BASELINE_VS_RATE_FP.csv


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/RATE_FP_Project")
RESULTS_DIR = BASE_DIR / "results"

all_results_path = RESULTS_DIR / "ALL_PROJECTS_RATE_FP_RESULTS.csv"
all_results_df = pd.read_csv(all_results_path)

def assign_family(model_name):
    if model_name in ["RATE-FP GraphSAGE ST", "RATE-FP Adaptive Hybrid Ensemble"]:
        return "Proposed RATE-FP"
    elif "ST" in model_name:
        return "Static + Temporal ML"
    else:
        return "Static-only ML"

all_results_df["Family"] = all_results_df["Model"].apply(assign_family)

summary_rows = []

for project in all_results_df["Project"].unique():
    project_df = all_results_df[all_results_df["Project"] == project].copy()

    static_best = (
        project_df[project_df["Family"] == "Static-only ML"]
        .sort_values("F1", ascending=False)
        .iloc[0]
    )

    temporal_best = (
        project_df[project_df["Family"] == "Static + Temporal ML"]
        .sort_values("F1", ascending=False)
        .iloc[0]
    )

    proposed_best = (
        project_df[project_df["Family"] == "Proposed RATE-FP"]
        .sort_values("F1", ascending=False)
        .iloc[0]
    )

    summary_rows.append({
        "Project": project,

        "Best_Static_Model": static_best["Model"],
        "Best_Static_F1": static_best["F1"],
        "Best_Static_ROC_AUC": static_best["ROC_AUC"],

        "Best_Temporal_Model": temporal_best["Model"],
        "Best_Temporal_F1": temporal_best["F1"],
        "Best_Temporal_ROC_AUC": temporal_best["ROC_AUC"],

        "Best_RATE_FP_Model": proposed_best["Model"],
        "Best_RATE_FP_F1": proposed_best["F1"],
        "Best_RATE_FP_ROC_AUC": proposed_best["ROC_AUC"],

        "F1_Gain_vs_Static": proposed_best["F1"] - static_best["F1"],
        "F1_Gain_vs_Temporal": proposed_best["F1"] - temporal_best["F1"],

        "ROC_Gain_vs_Static": proposed_best["ROC_AUC"] - static_best["ROC_AUC"],
        "ROC_Gain_vs_Temporal": proposed_best["ROC_AUC"] - temporal_best["ROC_AUC"],
    })

corrected_summary_df = pd.DataFrame(summary_rows)

avg_row = {
    "Project": "Average",
    "Best_Static_Model": "-",
    "Best_Static_F1": corrected_summary_df["Best_Static_F1"].mean(),
    "Best_Static_ROC_AUC": corrected_summary_df["Best_Static_ROC_AUC"].mean(),

    "Best_Temporal_Model": "-",
    "Best_Temporal_F1": corrected_summary_df["Best_Temporal_F1"].mean(),
    "Best_Temporal_ROC_AUC": corrected_summary_df["Best_Temporal_ROC_AUC"].mean(),

    "Best_RATE_FP_Model": "-",
    "Best_RATE_FP_F1": corrected_summary_df["Best_RATE_FP_F1"].mean(),
    "Best_RATE_FP_ROC_AUC": corrected_summary_df["Best_RATE_FP_ROC_AUC"].mean(),

    "F1_Gain_vs_Static": corrected_summary_df["F1_Gain_vs_Static"].mean(),
    "F1_Gain_vs_Temporal": corrected_summary_df["F1_Gain_vs_Temporal"].mean(),

    "ROC_Gain_vs_Static": corrected_summary_df["ROC_Gain_vs_Static"].mean(),
    "ROC_Gain_vs_Temporal": corrected_summary_df["ROC_Gain_vs_Temporal"].mean(),
}

corrected_summary_df = pd.concat(
    [corrected_summary_df, pd.DataFrame([avg_row])],
    ignore_index=True
)

display(corrected_summary_df)

corrected_summary_path = RESULTS_DIR / "CORRECTED_MODEL_FAMILY_SUMMARY.csv"
corrected_summary_df.to_csv(corrected_summary_path, index=False)

print("Saved corrected summary table to:")
print(corrected_summary_path)

,Project,Best_Static_Model,Best_Static_F1,Best_Static_ROC_AUC,Best_Temporal_Model,Best_Temporal_F1,Best_Temporal_ROC_AUC,Best_RATE_FP_Model,Best_RATE_FP_F1,Best_RATE_FP_ROC_AUC,F1_Gain_vs_Static,F1_Gain_vs_Temporal,ROC_Gain_vs_Static,ROC_Gain_vs_Temporal
0,jdt,GBR Static,0.620690,0.888588,LR ST,0.695652,0.887503,RATE-FP Adaptive Hybrid Ensemble,0.695652,0.915153,0.074963,0.000000,0.026565,0.027650
1,pde,SVC Static,0.457831,0.768540,LR ST,0.432990,0.750582,RATE-FP GraphSAGE ST,0.461538,0.822248,0.003707,0.028549,0.053708,0.071666
2,equinox,RF Static,0.682927,0.780702,SVC ST,0.619048,0.770175,RATE-FP GraphSAGE ST,0.571429,0.745614,-0.111498,-0.047619,-0.035088,-0.024561
3,lucene,LR Static,0.312500,0.639766,GBR ST,0.352941,0.705263,RATE-FP GraphSAGE ST,0.413793,0.750877,0.101293,0.060852,0.111111,0.045614
4,mylyn,RF Static,0.583333,0.835836,RF ST,0.461538,0.847181,RATE-FP Adaptive Hybrid Ensemble,0.483516,0.765210,-0.099817,0.021978,-0.070626,-0.081971
5,Average,-,0.531456,0.782686,-,0.512434,0.792141,-,0.525186,0.799820,-0.006270,0.012752,0.017134,0.007680


Saved corrected summary table to:
/content/drive/MyDrive/RATE_FP_Project/results/CORRECTED_MODEL_FAMILY_SUMMARY.csv


In [ ]:
from sklearn.metrics import f1_score
import numpy as np

def find_best_threshold(y_true, y_prob, start=0.10, end=0.90, step=0.01):
    thresholds = np.arange(start, end + step, step)
    best_threshold = 0.50
    best_f1 = -1

    for threshold in thresholds:
        pred = (y_prob >= threshold).astype(int)
        score = f1_score(y_true, pred, zero_division=0)

        if score > best_f1:
            best_f1 = score
            best_threshold = threshold

    return best_threshold, best_f1


def predict_with_threshold(y_prob, threshold):
    return (y_prob >= threshold).astype(int)

In [ ]:
def train_project_models_tuned(project_name, rate_fp_df, random_state=42):
    print("\n" + "="*120)
    print(f"Training tuned project: {project_name}")
    print("="*120)

    rate_fp_df = rate_fp_df.copy()

    for col in rate_fp_df.columns:
        if col != "classname":
            rate_fp_df[col] = pd.to_numeric(rate_fp_df[col], errors="coerce")

    rate_fp_df = rate_fp_df.fillna(0)
    rate_fp_df["is_defective"] = (rate_fp_df["bugs"] > 0).astype(int)

    target_cols = [
        "bugs", "nonTrivialBugs", "majorBugs",
        "criticalBugs", "highPriorityBugs", "is_defective"
    ]

    static_feature_cols = [
        col for col in rate_fp_df.columns
        if col not in ["classname"] + target_cols and "_temp_" not in col
    ]

    static_temporal_feature_cols = [
        col for col in rate_fp_df.columns
        if col not in ["classname"] + target_cols
    ]

    y = rate_fp_df["is_defective"].values
    indices_all = np.arange(len(rate_fp_df))

    train_idx, temp_idx, y_train, y_temp = train_test_split(
        indices_all,
        y,
        test_size=0.30,
        random_state=random_state,
        stratify=y
    )

    val_idx, test_idx, y_val, y_test = train_test_split(
        temp_idx,
        y_temp,
        test_size=0.50,
        random_state=random_state,
        stratify=y_temp
    )

    results = []

    model_val_f1_for_selector = {}
    model_test_result_for_selector = {}

    # ============================================================
    # 1. STATIC-ONLY MODELS WITH VALIDATION THRESHOLD TUNING
    # ============================================================
    X_static = rate_fp_df[static_feature_cols].values

    scaler_static = StandardScaler()
    X_static_train = scaler_static.fit_transform(X_static[train_idx])
    X_static_val = scaler_static.transform(X_static[val_idx])
    X_static_test = scaler_static.transform(X_static[test_idx])

    static_models = {
        "GBR Static Tuned": GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=random_state),
        "RF Static Tuned": RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced", random_state=random_state),
        "LR Static Tuned": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=random_state),
        "SVC Static Tuned": SVC(kernel="rbf", C=10, probability=True, class_weight="balanced", random_state=random_state),
        "KNN Static Tuned": KNeighborsClassifier(n_neighbors=5),
        "DT Static Tuned": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=random_state)
    }

    for model_name, model in static_models.items():
        start = time.time()

        model.fit(X_static_train, y[train_idx])

        val_prob = model.predict_proba(X_static_val)[:, 1]
        test_prob = model.predict_proba(X_static_test)[:, 1]

        best_threshold, best_val_f1 = find_best_threshold(y[val_idx], val_prob)
        test_pred = predict_with_threshold(test_prob, best_threshold)

        metrics = evaluate_predictions(y[test_idx], test_pred, test_prob)
        metrics.update({
            "Project": project_name,
            "Model": model_name,
            "Family": "Static-only ML",
            "Feature_Set": "Static",
            "Threshold": best_threshold,
            "Validation_F1": best_val_f1,
            "Training_Time_sec": time.time() - start
        })

        results.append(metrics)

    # ============================================================
    # 2. STATIC + TEMPORAL MODELS WITH VALIDATION THRESHOLD TUNING
    # ============================================================
    X_st = rate_fp_df[static_temporal_feature_cols].values

    scaler_st = StandardScaler()
    X_train = scaler_st.fit_transform(X_st[train_idx])
    X_val = scaler_st.transform(X_st[val_idx])
    X_test = scaler_st.transform(X_st[test_idx])

    st_models = {
        "RF ST Tuned": RandomForestClassifier(
            n_estimators=300, max_depth=10, class_weight="balanced", random_state=random_state
        ),
        "GBR ST Tuned": GradientBoostingClassifier(
            n_estimators=300, max_depth=3, random_state=random_state
        ),
        "SVC ST Tuned": SVC(
            kernel="rbf", C=10, probability=True, class_weight="balanced", random_state=random_state
        ),
        "LR ST Tuned": LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=random_state
        ),
        "KNN ST Tuned": KNeighborsClassifier(n_neighbors=5),
        "DT ST Tuned": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=random_state)
    }

    val_probs = {}
    test_probs = {}
    val_f1_scores = {}

    for model_name, model in st_models.items():
        start = time.time()

        model.fit(X_train, y[train_idx])

        val_prob = model.predict_proba(X_val)[:, 1]
        test_prob = model.predict_proba(X_test)[:, 1]

        best_threshold, best_val_f1 = find_best_threshold(y[val_idx], val_prob)
        test_pred = predict_with_threshold(test_prob, best_threshold)

        val_probs[model_name] = val_prob
        test_probs[model_name] = test_prob
        val_f1_scores[model_name] = best_val_f1

        metrics = evaluate_predictions(y[test_idx], test_pred, test_prob)
        metrics.update({
            "Project": project_name,
            "Model": model_name,
            "Family": "Static + Temporal ML",
            "Feature_Set": "Static + Temporal",
            "Threshold": best_threshold,
            "Validation_F1": best_val_f1,
            "Training_Time_sec": time.time() - start
        })

        results.append(metrics)

    # ============================================================
    # 3. GRAPHSAGE STATIC-TEMPORAL WITH VALIDATION THRESHOLD TUNING
    # ============================================================
    X_graph_scaled = StandardScaler().fit_transform(X_st)

    x_tensor = torch.tensor(X_graph_scaled, dtype=torch.float)
    y_tensor = torch.tensor(y, dtype=torch.long)

    edge_index = build_similarity_graph(X_graph_scaled, k_neighbors=10)

    num_nodes = x_tensor.shape[0]

    train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(num_nodes, dtype=torch.bool)

    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True

    data_st = Data(
        x=x_tensor,
        edge_index=edge_index,
        y=y_tensor,
        train_mask=train_mask,
        val_mask=val_mask,
        test_mask=test_mask
    ).to(device)

    graph_model = GraphSAGEDefectPredictor(
        in_channels=data_st.x.shape[1],
        hidden_channels=64,
        out_channels=2,
        dropout=0.40
    ).to(device)

    class_counts = np.bincount(y[train_idx])
    class_weights = class_counts.sum() / (2.0 * class_counts)
    class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

    optimizer = torch.optim.AdamW(graph_model.parameters(), lr=0.003, weight_decay=1e-3)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_val_f1 = 0
    best_state = None
    best_graph_threshold = 0.50
    patience = 50
    patience_counter = 0

    start_graph = time.time()

    for epoch in range(1, 401):
        graph_model.train()
        optimizer.zero_grad()

        logits = graph_model(data_st.x, data_st.edge_index)
        loss = criterion(logits[data_st.train_mask], data_st.y[data_st.train_mask])

        loss.backward()
        optimizer.step()

        graph_model.eval()
        with torch.no_grad():
            val_logits = graph_model(data_st.x, data_st.edge_index)
            val_prob_all = F.softmax(val_logits, dim=1)[:, 1].detach().cpu().numpy()
            val_prob = val_prob_all[val_idx]

        threshold, val_f1 = find_best_threshold(y[val_idx], val_prob)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_graph_threshold = threshold
            best_state = copy.deepcopy(graph_model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    if best_state is not None:
        graph_model.load_state_dict(best_state)

    graph_model.eval()
    with torch.no_grad():
        logits = graph_model(data_st.x, data_st.edge_index)
        graph_probs_all = F.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()

    graph_val_prob = graph_probs_all[val_idx]
    graph_test_prob = graph_probs_all[test_idx]

    graph_test_pred = predict_with_threshold(graph_test_prob, best_graph_threshold)

    graph_metrics = evaluate_predictions(y[test_idx], graph_test_pred, graph_test_prob)
    graph_metrics.update({
        "Project": project_name,
        "Model": "RATE-FP GraphSAGE ST Tuned",
        "Family": "Proposed RATE-FP",
        "Feature_Set": "Graph Static + Temporal",
        "Threshold": best_graph_threshold,
        "Validation_F1": best_val_f1,
        "Training_Time_sec": time.time() - start_graph
    })

    results.append(graph_metrics)

    val_probs["RATE-FP GraphSAGE ST Tuned"] = graph_val_prob
    test_probs["RATE-FP GraphSAGE ST Tuned"] = graph_test_prob
    val_f1_scores["RATE-FP GraphSAGE ST Tuned"] = best_val_f1

    model_val_f1_for_selector["RATE-FP GraphSAGE ST Tuned"] = best_val_f1
    model_test_result_for_selector["RATE-FP GraphSAGE ST Tuned"] = graph_metrics

    # ============================================================
    # 4. ADAPTIVE HYBRID ENSEMBLE WITH TUNED THRESHOLD
    # ============================================================
    selected_for_hybrid = [
        "RF ST Tuned",
        "GBR ST Tuned",
        "SVC ST Tuned",
        "LR ST Tuned",
        "RATE-FP GraphSAGE ST Tuned"
    ]

    raw_weights = np.array([val_f1_scores[m] for m in selected_for_hybrid], dtype=float) + 1e-6
    adaptive_weights = raw_weights / raw_weights.sum()

    final_val_prob = np.zeros_like(y[val_idx], dtype=float)
    final_test_prob = np.zeros_like(y[test_idx], dtype=float)

    for model_name, weight in zip(selected_for_hybrid, adaptive_weights):
        final_val_prob += weight * val_probs[model_name]
        final_test_prob += weight * test_probs[model_name]

    hybrid_threshold, hybrid_val_f1 = find_best_threshold(y[val_idx], final_val_prob)
    hybrid_test_pred = predict_with_threshold(final_test_prob, hybrid_threshold)

    hybrid_metrics = evaluate_predictions(y[test_idx], hybrid_test_pred, final_test_prob)
    hybrid_metrics.update({
        "Project": project_name,
        "Model": "RATE-FP Adaptive Hybrid Ensemble Tuned",
        "Family": "Proposed RATE-FP",
        "Feature_Set": "Hybrid Static + Temporal + Graph",
        "Threshold": hybrid_threshold,
        "Validation_F1": hybrid_val_f1,
        "Training_Time_sec": np.nan
    })

    results.append(hybrid_metrics)

    model_val_f1_for_selector["RATE-FP Adaptive Hybrid Ensemble Tuned"] = hybrid_val_f1
    model_test_result_for_selector["RATE-FP Adaptive Hybrid Ensemble Tuned"] = hybrid_metrics

    # ============================================================
    # 5. VALIDATION-GUIDED RATE-FP SELECTOR
    # ============================================================
    selected_rate_fp_model = max(
        model_val_f1_for_selector,
        key=model_val_f1_for_selector.get
    )

    selected_metrics = model_test_result_for_selector[selected_rate_fp_model].copy()
    selected_metrics["Model"] = "RATE-FP Validation-Guided Selector"
    selected_metrics["Selected_Model"] = selected_rate_fp_model
    selected_metrics["Family"] = "Proposed RATE-FP"
    selected_metrics["Feature_Set"] = "Validation-selected RATE-FP"
    selected_metrics["Training_Time_sec"] = np.nan

    results.append(selected_metrics)

    result_df = pd.DataFrame(results)

    tuned_project_result_path = RESULTS_DIR / f"{project_name}_TUNED_all_model_results.csv"
    result_df.to_csv(tuned_project_result_path, index=False)

    print(f"Completed tuned {project_name}")
    print("Selected RATE-FP model:", selected_rate_fp_model)
    print("Selected RATE-FP F1:", selected_metrics["F1"])
    print("Selected RATE-FP ROC-AUC:", selected_metrics["ROC_AUC"])
    print("Selected RATE-FP threshold:", selected_metrics["Threshold"])
    print("Confusion matrix:")
    print(confusion_matrix(y[test_idx], predict_with_threshold(
        test_probs.get(selected_rate_fp_model, final_test_prob),
        selected_metrics["Threshold"]
    )) if selected_rate_fp_model in test_probs else confusion_matrix(y[test_idx], hybrid_test_pred))

    return result_df

In [ ]:
tuned_all_project_results = []

for project_name, df in all_project_data.items():
    result_df = train_project_models_tuned(project_name, df, random_state=42)
    tuned_all_project_results.append(result_df)

tuned_all_results_df = pd.concat(tuned_all_project_results, ignore_index=True)

tuned_all_results_path = RESULTS_DIR / "ALL_PROJECTS_RATE_FP_TUNED_RESULTS.csv"
tuned_all_results_df.to_csv(tuned_all_results_path, index=False)

print("Saved tuned all-project results to:")
print(tuned_all_results_path)

display(tuned_all_results_df.sort_values(["Project", "F1"], ascending=[True, False]))


Training tuned project: jdt
Completed tuned jdt
Selected RATE-FP model: RATE-FP GraphSAGE ST Tuned
Selected RATE-FP F1: 0.75
Selected RATE-FP ROC-AUC: 0.8950935212794796
Selected RATE-FP threshold: 0.7599999999999997
Confusion matrix:
[[110   9]
 [  7  24]]

Training tuned project: pde
Completed tuned pde
Selected RATE-FP model: RATE-FP GraphSAGE ST Tuned
Selected RATE-FP F1: 0.4594594594594595
Selected RATE-FP ROC-AUC: 0.8037911539740605
Selected RATE-FP threshold: 0.6399999999999997
Confusion matrix:
[[168  26]
 [ 14  17]]

Training tuned project: equinox
Completed tuned equinox
Selected RATE-FP model: RATE-FP Adaptive Hybrid Ensemble Tuned
Selected RATE-FP F1: 0.5294117647058824
Selected RATE-FP ROC-AUC: 0.7736842105263159
Selected RATE-FP threshold: 0.5899999999999997
Confusion matrix:
[[24  6]
 [10  9]]

Training tuned project: lucene
Completed tuned lucene
Selected RATE-FP model: RATE-FP GraphSAGE ST Tuned
Selected RATE-FP F1: 0.2608695652173913
Selected RATE-FP ROC-AUC: 0.70292

,Accuracy,Precision,Recall,F1,ROC_AUC,Project,Model,Family,Feature_Set,Threshold,Validation_F1,Training_Time_sec,Selected_Model
30,0.734694,0.636364,0.736842,0.682927,0.782456,equinox,GBR Static Tuned,Static-only ML,Static,0.40,0.857143,0.567715,NaN
33,0.714286,0.600000,0.789474,0.681818,0.728070,equinox,SVC Static Tuned,Static-only ML,Static,0.37,0.790698,0.146324,NaN
31,0.673469,0.555556,0.789474,0.652174,0.780702,equinox,RF Static Tuned,Static-only ML,Static,0.44,0.837209,0.473545,NaN
42,0.673469,0.555556,0.789474,0.652174,0.771930,equinox,RATE-FP GraphSAGE ST Tuned,Proposed RATE-FP,Graph Static + Temporal,0.50,0.826087,8.199600,NaN
38,0.693878,0.583333,0.736842,0.651163,0.770175,equinox,SVC ST Tuned,Static + Temporal ML,Static + Temporal,0.44,0.761905,0.160874,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18,0.586667,0.223214,0.806452,0.349650,0.768540,pde,SVC Static Tuned,Static-only ML,Static,0.13,0.317241,0.382944,NaN
15,0.746667,0.267857,0.483871,0.344828,0.772531,pde,GBR Static Tuned,Static-only ML,Static,0.14,0.449438,1.470378,NaN
23,0.773333,0.272727,0.387097,0.320000,0.730961,pde,SVC ST Tuned,Static + Temporal ML,Static + Temporal,0.19,0.347826,1.148232,NaN
25,0.635556,0.213483,0.612903,0.316667,0.650482,pde,KNN ST Tuned,Static + Temporal ML,Static + Temporal,0.10,0.375940,0.258480,NaN


In [ ]:
summary_rows = []

for project in tuned_all_results_df["Project"].unique():
    project_df = tuned_all_results_df[tuned_all_results_df["Project"] == project].copy()

    static_best = (
        project_df[project_df["Family"] == "Static-only ML"]
        .sort_values("F1", ascending=False)
        .iloc[0]
    )

    temporal_best = (
        project_df[project_df["Family"] == "Static + Temporal ML"]
        .sort_values("F1", ascending=False)
        .iloc[0]
    )

    proposed_selector = (
        project_df[project_df["Model"] == "RATE-FP Validation-Guided Selector"]
        .iloc[0]
    )

    summary_rows.append({
        "Project": project,

        "Best_Static_Model": static_best["Model"],
        "Best_Static_F1": static_best["F1"],
        "Best_Static_ROC_AUC": static_best["ROC_AUC"],

        "Best_Temporal_Model": temporal_best["Model"],
        "Best_Temporal_F1": temporal_best["F1"],
        "Best_Temporal_ROC_AUC": temporal_best["ROC_AUC"],

        "Selected_RATE_FP_Model": proposed_selector.get("Selected_Model", "-"),
        "RATE_FP_F1": proposed_selector["F1"],
        "RATE_FP_ROC_AUC": proposed_selector["ROC_AUC"],
        "RATE_FP_Accuracy": proposed_selector["Accuracy"],
        "RATE_FP_Precision": proposed_selector["Precision"],
        "RATE_FP_Recall": proposed_selector["Recall"],

        "F1_Gain_vs_Static": proposed_selector["F1"] - static_best["F1"],
        "F1_Gain_vs_Temporal": proposed_selector["F1"] - temporal_best["F1"],

        "ROC_Gain_vs_Static": proposed_selector["ROC_AUC"] - static_best["ROC_AUC"],
        "ROC_Gain_vs_Temporal": proposed_selector["ROC_AUC"] - temporal_best["ROC_AUC"],
    })

final_tuned_summary_df = pd.DataFrame(summary_rows)

avg_row = {
    "Project": "Average",

    "Best_Static_Model": "-",
    "Best_Static_F1": final_tuned_summary_df["Best_Static_F1"].mean(),
    "Best_Static_ROC_AUC": final_tuned_summary_df["Best_Static_ROC_AUC"].mean(),

    "Best_Temporal_Model": "-",
    "Best_Temporal_F1": final_tuned_summary_df["Best_Temporal_F1"].mean(),
    "Best_Temporal_ROC_AUC": final_tuned_summary_df["Best_Temporal_ROC_AUC"].mean(),

    "Selected_RATE_FP_Model": "-",
    "RATE_FP_F1": final_tuned_summary_df["RATE_FP_F1"].mean(),
    "RATE_FP_ROC_AUC": final_tuned_summary_df["RATE_FP_ROC_AUC"].mean(),
    "RATE_FP_Accuracy": final_tuned_summary_df["RATE_FP_Accuracy"].mean(),
    "RATE_FP_Precision": final_tuned_summary_df["RATE_FP_Precision"].mean(),
    "RATE_FP_Recall": final_tuned_summary_df["RATE_FP_Recall"].mean(),

    "F1_Gain_vs_Static": final_tuned_summary_df["F1_Gain_vs_Static"].mean(),
    "F1_Gain_vs_Temporal": final_tuned_summary_df["F1_Gain_vs_Temporal"].mean(),

    "ROC_Gain_vs_Static": final_tuned_summary_df["ROC_Gain_vs_Static"].mean(),
    "ROC_Gain_vs_Temporal": final_tuned_summary_df["ROC_Gain_vs_Temporal"].mean(),
}

final_tuned_summary_df = pd.concat(
    [final_tuned_summary_df, pd.DataFrame([avg_row])],
    ignore_index=True
)

display(final_tuned_summary_df)

final_tuned_summary_path = RESULTS_DIR / "FINAL_TUNED_RATE_FP_SUMMARY.csv"
final_tuned_summary_df.to_csv(final_tuned_summary_path, index=False)

print("Saved final tuned RATE-FP summary to:")
print(final_tuned_summary_path)

,Project,Best_Static_Model,Best_Static_F1,Best_Static_ROC_AUC,Best_Temporal_Model,Best_Temporal_F1,Best_Temporal_ROC_AUC,Selected_RATE_FP_Model,RATE_FP_F1,RATE_FP_ROC_AUC,RATE_FP_Accuracy,RATE_FP_Precision,RATE_FP_Recall,F1_Gain_vs_Static,F1_Gain_vs_Temporal,ROC_Gain_vs_Static,ROC_Gain_vs_Temporal
0,jdt,GBR Static Tuned,0.686567,0.888588,RF ST Tuned,0.704225,0.898346,RATE-FP GraphSAGE ST Tuned,0.750000,0.895094,0.893333,0.727273,0.774194,0.063433,0.045775,0.006506,-0.003253
1,pde,RF Static Tuned,0.477612,0.803126,GBR ST Tuned,0.441176,0.808447,RATE-FP GraphSAGE ST Tuned,0.459459,0.803791,0.822222,0.395349,0.548387,-0.018152,0.018283,0.000665,-0.004656
2,equinox,GBR Static Tuned,0.682927,0.782456,SVC ST Tuned,0.651163,0.770175,RATE-FP Adaptive Hybrid Ensemble Tuned,0.529412,0.773684,0.673469,0.600000,0.473684,-0.153515,-0.121751,-0.008772,0.003509
3,lucene,RF Static Tuned,0.250000,0.700585,GBR ST Tuned,0.285714,0.705263,RATE-FP GraphSAGE ST Tuned,0.260870,0.702924,0.836538,0.214286,0.333333,0.010870,-0.024845,0.002339,-0.002339
4,mylyn,RF Static Tuned,0.536585,0.835836,RF ST Tuned,0.495868,0.847181,RATE-FP GraphSAGE ST Tuned,0.452174,0.803915,0.775000,0.333333,0.702703,-0.084411,-0.043694,-0.031921,-0.043265
5,Average,-,0.526738,0.802118,-,0.515629,0.805883,-,0.490383,0.795882,0.800113,0.454048,0.566460,-0.036355,-0.025246,-0.006237,-0.010001


Saved final tuned RATE-FP summary to:
/content/drive/MyDrive/RATE_FP_Project/results/FINAL_TUNED_RATE_FP_SUMMARY.csv


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/RATE_FP_Project")
RESULTS_DIR = BASE_DIR / "results"

tuned_path = RESULTS_DIR / "ALL_PROJECTS_RATE_FP_TUNED_RESULTS.csv"
tuned_all_results_df = pd.read_csv(tuned_path)

robust_rows = []

for project in tuned_all_results_df["Project"].unique():
    project_df = tuned_all_results_df[tuned_all_results_df["Project"] == project].copy()

    # Remove previous selector row to avoid selecting a selector
    candidate_df = project_df[
        project_df["Model"] != "RATE-FP Validation-Guided Selector"
    ].copy()

    # Select model using validation F1 only
    selected = candidate_df.sort_values("Validation_F1", ascending=False).iloc[0]

    # Also compute best observed test result per family for analysis only
    best_static = (
        project_df[project_df["Family"] == "Static-only ML"]
        .sort_values("F1", ascending=False)
        .iloc[0]
    )

    best_temporal = (
        project_df[project_df["Family"] == "Static + Temporal ML"]
        .sort_values("F1", ascending=False)
        .iloc[0]
    )

    proposed_candidates = project_df[
        project_df["Family"] == "Proposed RATE-FP"
    ]

    best_ratefp_component = (
        proposed_candidates
        .sort_values("F1", ascending=False)
        .iloc[0]
    )

    robust_rows.append({
        "Project": project,

        "Selected_Model_By_Validation": selected["Model"],
        "Selected_Family": selected["Family"],
        "Selected_Validation_F1": selected["Validation_F1"],

        "Robust_RATE_FP_Accuracy": selected["Accuracy"],
        "Robust_RATE_FP_Precision": selected["Precision"],
        "Robust_RATE_FP_Recall": selected["Recall"],
        "Robust_RATE_FP_F1": selected["F1"],
        "Robust_RATE_FP_ROC_AUC": selected["ROC_AUC"],
        "Robust_RATE_FP_Threshold": selected["Threshold"],

        "Best_Static_Model": best_static["Model"],
        "Best_Static_F1": best_static["F1"],
        "Best_Static_ROC_AUC": best_static["ROC_AUC"],

        "Best_Temporal_Model": best_temporal["Model"],
        "Best_Temporal_F1": best_temporal["F1"],
        "Best_Temporal_ROC_AUC": best_temporal["ROC_AUC"],

        "Best_RATEFP_Component": best_ratefp_component["Model"],
        "Best_RATEFP_Component_F1": best_ratefp_component["F1"],
        "Best_RATEFP_Component_ROC_AUC": best_ratefp_component["ROC_AUC"],

        "F1_Gain_vs_Static": selected["F1"] - best_static["F1"],
        "F1_Gain_vs_Temporal": selected["F1"] - best_temporal["F1"],
        "ROC_Gain_vs_Static": selected["ROC_AUC"] - best_static["ROC_AUC"],
        "ROC_Gain_vs_Temporal": selected["ROC_AUC"] - best_temporal["ROC_AUC"]
    })

robust_summary_df = pd.DataFrame(robust_rows)

avg_row = {
    "Project": "Average",
    "Selected_Model_By_Validation": "-",
    "Selected_Family": "-",
    "Selected_Validation_F1": robust_summary_df["Selected_Validation_F1"].mean(),

    "Robust_RATE_FP_Accuracy": robust_summary_df["Robust_RATE_FP_Accuracy"].mean(),
    "Robust_RATE_FP_Precision": robust_summary_df["Robust_RATE_FP_Precision"].mean(),
    "Robust_RATE_FP_Recall": robust_summary_df["Robust_RATE_FP_Recall"].mean(),
    "Robust_RATE_FP_F1": robust_summary_df["Robust_RATE_FP_F1"].mean(),
    "Robust_RATE_FP_ROC_AUC": robust_summary_df["Robust_RATE_FP_ROC_AUC"].mean(),
    "Robust_RATE_FP_Threshold": robust_summary_df["Robust_RATE_FP_Threshold"].mean(),

    "Best_Static_Model": "-",
    "Best_Static_F1": robust_summary_df["Best_Static_F1"].mean(),
    "Best_Static_ROC_AUC": robust_summary_df["Best_Static_ROC_AUC"].mean(),

    "Best_Temporal_Model": "-",
    "Best_Temporal_F1": robust_summary_df["Best_Temporal_F1"].mean(),
    "Best_Temporal_ROC_AUC": robust_summary_df["Best_Temporal_ROC_AUC"].mean(),

    "Best_RATEFP_Component": "-",
    "Best_RATEFP_Component_F1": robust_summary_df["Best_RATEFP_Component_F1"].mean(),
    "Best_RATEFP_Component_ROC_AUC": robust_summary_df["Best_RATEFP_Component_ROC_AUC"].mean(),

    "F1_Gain_vs_Static": robust_summary_df["F1_Gain_vs_Static"].mean(),
    "F1_Gain_vs_Temporal": robust_summary_df["F1_Gain_vs_Temporal"].mean(),
    "ROC_Gain_vs_Static": robust_summary_df["ROC_Gain_vs_Static"].mean(),
    "ROC_Gain_vs_Temporal": robust_summary_df["ROC_Gain_vs_Temporal"].mean()
}

robust_summary_df = pd.concat(
    [robust_summary_df, pd.DataFrame([avg_row])],
    ignore_index=True
)

display(robust_summary_df)

robust_summary_path = RESULTS_DIR / "ROBUST_RATE_FP_SELECTOR_SUMMARY.csv"
robust_summary_df.to_csv(robust_summary_path, index=False)

print("Saved robust RATE-FP selector summary to:")
print(robust_summary_path)

,Project,Selected_Model_By_Validation,Selected_Family,Selected_Validation_F1,Robust_RATE_FP_Accuracy,Robust_RATE_FP_Precision,Robust_RATE_FP_Recall,Robust_RATE_FP_F1,Robust_RATE_FP_ROC_AUC,Robust_RATE_FP_Threshold,...,Best_Temporal_Model,Best_Temporal_F1,Best_Temporal_ROC_AUC,Best_RATEFP_Component,Best_RATEFP_Component_F1,Best_RATEFP_Component_ROC_AUC,F1_Gain_vs_Static,F1_Gain_vs_Temporal,ROC_Gain_vs_Static,ROC_Gain_vs_Temporal
0,jdt,RATE-FP GraphSAGE ST Tuned,Proposed RATE-FP,0.666667,0.893333,0.727273,0.774194,0.750000,0.895094,0.760,...,RF ST Tuned,0.704225,0.898346,RATE-FP GraphSAGE ST Tuned,0.750000,0.895094,0.063433,0.045775,0.006506,-0.003253
1,pde,RF ST Tuned,Static + Temporal ML,0.540541,0.791111,0.346154,0.580645,0.433735,0.807782,0.290,...,GBR ST Tuned,0.441176,0.808447,RATE-FP GraphSAGE ST Tuned,0.459459,0.803791,-0.043877,-0.007442,0.004656,-0.000665
2,equinox,GBR Static Tuned,Static-only ML,0.857143,0.734694,0.636364,0.736842,0.682927,0.782456,0.400,...,SVC ST Tuned,0.651163,0.770175,RATE-FP GraphSAGE ST Tuned,0.652174,0.771930,0.000000,0.031764,0.000000,0.012281
3,lucene,RATE-FP GraphSAGE ST Tuned,Proposed RATE-FP,0.631579,0.836538,0.214286,0.333333,0.260870,0.702924,0.710,...,GBR ST Tuned,0.285714,0.705263,RATE-FP Adaptive Hybrid Ensemble Tuned,0.285714,0.730994,0.010870,-0.024845,0.002339,-0.002339
4,mylyn,RF Static Tuned,Static-only ML,0.518519,0.864286,0.488889,0.594595,0.536585,0.835836,0.450,...,RF ST Tuned,0.495868,0.847181,RATE-FP Adaptive Hybrid Ensemble Tuned,0.464286,0.833945,0.000000,0.040718,0.000000,-0.011345
5,Average,-,-,0.642890,0.823992,0.482593,0.603922,0.532823,0.804818,0.522,...,-,0.515629,0.805883,-,0.522327,0.807151,0.006085,0.017194,0.002700,-0.001064


Saved robust RATE-FP selector summary to:
/content/drive/MyDrive/RATE_FP_Project/results/ROBUST_RATE_FP_SELECTOR_SUMMARY.csv


In [ ]:
seeds = [21, 42, 84, 126, 168]

multi_seed_results = []

for seed in seeds:
    print("\n" + "#"*120)
    print(f"Running experiments for seed: {seed}")
    print("#"*120)

    seed_project_results = []

    for project_name, df in all_project_data.items():
        result_df = train_project_models_tuned(project_name, df, random_state=seed)
        result_df["Seed"] = seed
        seed_project_results.append(result_df)

    seed_all_df = pd.concat(seed_project_results, ignore_index=True)

    # Robust selector for this seed
    for project in seed_all_df["Project"].unique():
        project_df = seed_all_df[seed_all_df["Project"] == project].copy()

        candidate_df = project_df[
            project_df["Model"] != "RATE-FP Validation-Guided Selector"
        ].copy()

        selected = candidate_df.sort_values("Validation_F1", ascending=False).iloc[0]

        best_static = (
            project_df[project_df["Family"] == "Static-only ML"]
            .sort_values("F1", ascending=False)
            .iloc[0]
        )

        best_temporal = (
            project_df[project_df["Family"] == "Static + Temporal ML"]
            .sort_values("F1", ascending=False)
            .iloc[0]
        )

        proposed_components = project_df[
            project_df["Family"] == "Proposed RATE-FP"
        ]

        best_ratefp_component = (
            proposed_components
            .sort_values("F1", ascending=False)
            .iloc[0]
        )

        multi_seed_results.append({
            "Seed": seed,
            "Project": project,

            "Selected_Model_By_Validation": selected["Model"],
            "Selected_Family": selected["Family"],

            "Best_Static_F1": best_static["F1"],
            "Best_Static_ROC_AUC": best_static["ROC_AUC"],
            "Best_Static_Recall": best_static["Recall"],

            "Best_Temporal_F1": best_temporal["F1"],
            "Best_Temporal_ROC_AUC": best_temporal["ROC_AUC"],
            "Best_Temporal_Recall": best_temporal["Recall"],

            "Best_RATEFP_Component_F1": best_ratefp_component["F1"],
            "Best_RATEFP_Component_ROC_AUC": best_ratefp_component["ROC_AUC"],
            "Best_RATEFP_Component_Recall": best_ratefp_component["Recall"],

            "Robust_RATE_FP_F1": selected["F1"],
            "Robust_RATE_FP_ROC_AUC": selected["ROC_AUC"],
            "Robust_RATE_FP_Accuracy": selected["Accuracy"],
            "Robust_RATE_FP_Precision": selected["Precision"],
            "Robust_RATE_FP_Recall": selected["Recall"],

            "F1_Gain_vs_Static": selected["F1"] - best_static["F1"],
            "F1_Gain_vs_Temporal": selected["F1"] - best_temporal["F1"],
            "ROC_Gain_vs_Static": selected["ROC_AUC"] - best_static["ROC_AUC"],
            "ROC_Gain_vs_Temporal": selected["ROC_AUC"] - best_temporal["ROC_AUC"]
        })

multi_seed_df = pd.DataFrame(multi_seed_results)

multi_seed_path = RESULTS_DIR / "MULTI_SEED_ROBUST_RATE_FP_RESULTS.csv"
multi_seed_df.to_csv(multi_seed_path, index=False)

print("Saved multi-seed robust RATE-FP results to:")
print(multi_seed_path)

display(multi_seed_df)


########################################################################################################################
Running experiments for seed: 21
########################################################################################################################

Training tuned project: jdt
Completed tuned jdt
Selected RATE-FP model: RATE-FP GraphSAGE ST Tuned
Selected RATE-FP F1: 0.6376811594202898
Selected RATE-FP ROC-AUC: 0.8468419625914881
Selected RATE-FP threshold: 0.47999999999999976
Confusion matrix:
[[103  16]
 [  9  22]]

Training tuned project: pde
Completed tuned pde
Selected RATE-FP model: RATE-FP GraphSAGE ST Tuned
Selected RATE-FP F1: 0.35714285714285715
Selected RATE-FP ROC-AUC: 0.7906551380113069
Selected RATE-FP threshold: 0.6599999999999997
Confusion matrix:
[[179  15]
 [ 21  10]]

Training tuned project: equinox
Completed tuned equinox
Selected RATE-FP model: RATE-FP GraphSAGE ST Tuned
Selected RATE-FP F1: 0.6976744186046512
Selected RATE-FP ROC-AUC: 0.

,Seed,Project,Selected_Model_By_Validation,Selected_Family,Best_Static_F1,Best_Static_ROC_AUC,Best_Static_Recall,Best_Temporal_F1,Best_Temporal_ROC_AUC,Best_Temporal_Recall,...,Best_RATEFP_Component_Recall,Robust_RATE_FP_F1,Robust_RATE_FP_ROC_AUC,Robust_RATE_FP_Accuracy,Robust_RATE_FP_Precision,Robust_RATE_FP_Recall,F1_Gain_vs_Static,F1_Gain_vs_Temporal,ROC_Gain_vs_Static,ROC_Gain_vs_Temporal
0,21,jdt,RF ST Tuned,Static + Temporal ML,0.615385,0.735430,0.645161,0.647887,0.834101,0.741935,...,0.806452,0.637681,0.887774,0.833333,0.578947,0.709677,0.022297,-0.010206,0.152345,0.053673
1,21,pde,RATE-FP GraphSAGE ST Tuned,Proposed RATE-FP,0.385965,0.765381,0.354839,0.451613,0.796807,0.451613,...,0.612903,0.357143,0.790655,0.840000,0.400000,0.322581,-0.028822,-0.094470,0.025274,-0.006152
2,21,equinox,RATE-FP GraphSAGE ST Tuned,Proposed RATE-FP,0.717949,0.801724,0.700000,0.756757,0.875862,0.700000,...,0.750000,0.697674,0.855172,0.734694,0.652174,0.750000,-0.020274,-0.059082,0.053448,-0.020690
3,21,lucene,RATE-FP Adaptive Hybrid Ensemble Tuned,Proposed RATE-FP,0.434783,0.791489,0.500000,0.470588,0.818085,0.400000,...,0.400000,0.363636,0.875532,0.865385,0.333333,0.400000,-0.071146,-0.106952,0.084043,0.057447
4,21,mylyn,RATE-FP GraphSAGE ST Tuned,Proposed RATE-FP,0.480000,0.811589,0.648649,0.512821,0.830720,0.540541,...,0.648649,0.351351,0.749194,0.828571,0.351351,0.351351,-0.128649,-0.161469,-0.062396,-0.081526
5,42,jdt,RATE-FP GraphSAGE ST Tuned,Proposed RATE-FP,0.686567,0.888588,0.741935,0.704225,0.898346,0.806452,...,0.612903,0.644068,0.900244,0.860000,0.678571,0.612903,-0.042499,-0.060158,0.011656,0.001898
6,42,pde,RF ST Tuned,Static + Temporal ML,0.477612,0.803126,0.516129,0.441176,0.808447,0.483871,...,0.516129,0.433735,0.807782,0.791111,0.346154,0.580645,-0.043877,-0.007442,0.004656,-0.000665
7,42,equinox,GBR Static Tuned,Static-only ML,0.682927,0.782456,0.736842,0.651163,0.770175,0.736842,...,0.894737,0.682927,0.782456,0.734694,0.636364,0.736842,0.000000,0.031764,0.000000,0.012281
8,42,lucene,RATE-FP GraphSAGE ST Tuned,Proposed RATE-FP,0.250000,0.700585,0.333333,0.285714,0.705263,0.333333,...,0.555556,0.277778,0.661988,0.750000,0.185185,0.555556,0.027778,-0.007937,-0.038596,-0.043275
9,42,mylyn,RF Static Tuned,Static-only ML,0.536585,0.835836,0.594595,0.495868,0.847181,0.810811,...,0.756757,0.536585,0.835836,0.864286,0.488889,0.594595,0.000000,0.040718,0.000000,-0.011345


In [ ]:
project_summary = multi_seed_df.groupby("Project").agg({
    "Best_Static_F1": ["mean", "std"],
    "Best_Temporal_F1": ["mean", "std"],
    "Robust_RATE_FP_F1": ["mean", "std"],

    "Best_Static_ROC_AUC": ["mean", "std"],
    "Best_Temporal_ROC_AUC": ["mean", "std"],
    "Robust_RATE_FP_ROC_AUC": ["mean", "std"],

    "Best_Static_Recall": ["mean", "std"],
    "Best_Temporal_Recall": ["mean", "std"],
    "Robust_RATE_FP_Recall": ["mean", "std"],

    "F1_Gain_vs_Static": ["mean", "std"],
    "F1_Gain_vs_Temporal": ["mean", "std"]
}).reset_index()

# Flatten column names
project_summary.columns = [
    "_".join(col).strip("_") for col in project_summary.columns.values
]

display(project_summary)

project_summary_path = RESULTS_DIR / "MULTI_SEED_PROJECT_SUMMARY.csv"
project_summary.to_csv(project_summary_path, index=False)

print("Saved project-wise multi-seed summary to:")
print(project_summary_path)

,Project,Best_Static_F1_mean,Best_Static_F1_std,Best_Temporal_F1_mean,Best_Temporal_F1_std,Robust_RATE_FP_F1_mean,Robust_RATE_FP_F1_std,Best_Static_ROC_AUC_mean,Best_Static_ROC_AUC_std,Best_Temporal_ROC_AUC_mean,...,Best_Static_Recall_mean,Best_Static_Recall_std,Best_Temporal_Recall_mean,Best_Temporal_Recall_std,Robust_RATE_FP_Recall_mean,Robust_RATE_FP_Recall_std,F1_Gain_vs_Static_mean,F1_Gain_vs_Static_std,F1_Gain_vs_Temporal_mean,F1_Gain_vs_Temporal_std
0,equinox,0.711602,0.028172,0.709046,0.055430,0.645947,0.093116,0.825675,0.034460,0.810094,...,0.721579,0.027999,0.762105,0.136609,0.669474,0.147915,-0.065655,0.087171,-0.063098,0.070323
1,jdt,0.605006,0.054956,0.645693,0.040572,0.600813,0.050538,0.816753,0.059463,0.858037,...,0.677419,0.045620,0.696774,0.077688,0.612903,0.088342,-0.004194,0.056155,-0.044880,0.050936
2,lucene,0.394032,0.099592,0.387473,0.113864,0.315729,0.068422,0.768349,0.085366,0.778193,...,0.468889,0.126296,0.511111,0.237788,0.540000,0.273026,-0.078303,0.068499,-0.071744,0.050230
3,mylyn,0.489091,0.027943,0.454478,0.046408,0.431304,0.081702,0.810121,0.030492,0.778412,...,0.610811,0.052685,0.589189,0.177640,0.481081,0.092447,-0.057787,0.060282,-0.023175,0.095068
4,pde,0.436120,0.034358,0.444183,0.030139,0.402110,0.030346,0.774110,0.017514,0.780845,...,0.529032,0.124099,0.483871,0.039508,0.425806,0.095148,-0.034010,0.008303,-0.042073,0.041573


Saved project-wise multi-seed summary to:
/content/drive/MyDrive/RATE_FP_Project/results/MULTI_SEED_PROJECT_SUMMARY.csv


In [ ]:
overall_summary = pd.DataFrame({
    "Metric": [
        "Static-only F1",
        "Static + Temporal F1",
        "Robust RATE-FP F1",

        "Static-only ROC-AUC",
        "Static + Temporal ROC-AUC",
        "Robust RATE-FP ROC-AUC",

        "Static-only Recall",
        "Static + Temporal Recall",
        "Robust RATE-FP Recall",

        "F1 Gain vs Static",
        "F1 Gain vs Temporal"
    ],
    "Mean": [
        multi_seed_df["Best_Static_F1"].mean(),
        multi_seed_df["Best_Temporal_F1"].mean(),
        multi_seed_df["Robust_RATE_FP_F1"].mean(),

        multi_seed_df["Best_Static_ROC_AUC"].mean(),
        multi_seed_df["Best_Temporal_ROC_AUC"].mean(),
        multi_seed_df["Robust_RATE_FP_ROC_AUC"].mean(),

        multi_seed_df["Best_Static_Recall"].mean(),
        multi_seed_df["Best_Temporal_Recall"].mean(),
        multi_seed_df["Robust_RATE_FP_Recall"].mean(),

        multi_seed_df["F1_Gain_vs_Static"].mean(),
        multi_seed_df["F1_Gain_vs_Temporal"].mean()
    ],
    "Std": [
        multi_seed_df["Best_Static_F1"].std(),
        multi_seed_df["Best_Temporal_F1"].std(),
        multi_seed_df["Robust_RATE_FP_F1"].std(),

        multi_seed_df["Best_Static_ROC_AUC"].std(),
        multi_seed_df["Best_Temporal_ROC_AUC"].std(),
        multi_seed_df["Robust_RATE_FP_ROC_AUC"].std(),

        multi_seed_df["Best_Static_Recall"].std(),
        multi_seed_df["Best_Temporal_Recall"].std(),
        multi_seed_df["Robust_RATE_FP_Recall"].std(),

        multi_seed_df["F1_Gain_vs_Static"].std(),
        multi_seed_df["F1_Gain_vs_Temporal"].std()
    ]
})

display(overall_summary)

overall_summary_path = RESULTS_DIR / "MULTI_SEED_OVERALL_SUMMARY.csv"
overall_summary.to_csv(overall_summary_path, index=False)

print("Saved overall multi-seed summary to:")
print(overall_summary_path)

,Metric,Mean,Std
0,Static-only F1,0.527170,0.129176
1,Static + Temporal F1,0.528175,0.140982
2,Robust RATE-FP F1,0.479180,0.141683
3,Static-only ROC-AUC,0.799002,0.052653
4,Static + Temporal ROC-AUC,0.801116,0.066154
5,Robust RATE-FP ROC-AUC,0.801059,0.081119
6,Static-only Recall,0.601546,0.123069
7,Static + Temporal Recall,0.608610,0.175750
8,Robust RATE-FP Recall,0.545853,0.168250
9,F1 Gain vs Static,-0.047990,0.062514


Saved overall multi-seed summary to:
/content/drive/MyDrive/RATE_FP_Project/results/MULTI_SEED_OVERALL_SUMMARY.csv


In [ ]:
def train_project_models_graph_risk_augmented(project_name, rate_fp_df, random_state=42):
    print("\n" + "="*120)
    print(f"Training Graph-Risk Augmented RATE-FP project: {project_name}")
    print("="*120)

    rate_fp_df = rate_fp_df.copy()

    for col in rate_fp_df.columns:
        if col != "classname":
            rate_fp_df[col] = pd.to_numeric(rate_fp_df[col], errors="coerce")

    rate_fp_df = rate_fp_df.fillna(0)
    rate_fp_df["is_defective"] = (rate_fp_df["bugs"] > 0).astype(int)

    target_cols = [
        "bugs", "nonTrivialBugs", "majorBugs",
        "criticalBugs", "highPriorityBugs", "is_defective"
    ]

    static_feature_cols = [
        col for col in rate_fp_df.columns
        if col not in ["classname"] + target_cols and "_temp_" not in col
    ]

    static_temporal_feature_cols = [
        col for col in rate_fp_df.columns
        if col not in ["classname"] + target_cols
    ]

    y = rate_fp_df["is_defective"].values
    indices_all = np.arange(len(rate_fp_df))

    train_idx, temp_idx, y_train, y_temp = train_test_split(
        indices_all,
        y,
        test_size=0.30,
        random_state=random_state,
        stratify=y
    )

    val_idx, test_idx, y_val, y_test = train_test_split(
        temp_idx,
        y_temp,
        test_size=0.50,
        random_state=random_state,
        stratify=y_temp
    )

    results = []

    # ==========================================================
    # 1. Static-only baseline
    # ==========================================================
    X_static = rate_fp_df[static_feature_cols].values

    scaler_static = StandardScaler()
    X_static_train = scaler_static.fit_transform(X_static[train_idx])
    X_static_val = scaler_static.transform(X_static[val_idx])
    X_static_test = scaler_static.transform(X_static[test_idx])

    static_models = {
        "GBR Static Tuned": GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=random_state),
        "RF Static Tuned": RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced", random_state=random_state),
        "LR Static Tuned": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=random_state),
        "SVC Static Tuned": SVC(kernel="rbf", C=10, probability=True, class_weight="balanced", random_state=random_state),
        "KNN Static Tuned": KNeighborsClassifier(n_neighbors=5),
        "DT Static Tuned": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=random_state)
    }

    for model_name, model in static_models.items():
        start = time.time()

        model.fit(X_static_train, y[train_idx])

        val_prob = model.predict_proba(X_static_val)[:, 1]
        test_prob = model.predict_proba(X_static_test)[:, 1]

        threshold, val_f1 = find_best_threshold(y[val_idx], val_prob)
        test_pred = predict_with_threshold(test_prob, threshold)

        metrics = evaluate_predictions(y[test_idx], test_pred, test_prob)
        metrics.update({
            "Project": project_name,
            "Model": model_name,
            "Family": "Static-only ML",
            "Feature_Set": "Static",
            "Threshold": threshold,
            "Validation_F1": val_f1,
            "Training_Time_sec": time.time() - start
        })

        results.append(metrics)

    # ==========================================================
    # 2. Static + Temporal baseline
    # ==========================================================
    X_st = rate_fp_df[static_temporal_feature_cols].values

    scaler_st = StandardScaler()
    X_train = scaler_st.fit_transform(X_st[train_idx])
    X_val = scaler_st.transform(X_st[val_idx])
    X_test = scaler_st.transform(X_st[test_idx])

    temporal_models = {
        "RF ST Tuned": RandomForestClassifier(
            n_estimators=300, max_depth=10, class_weight="balanced", random_state=random_state
        ),
        "GBR ST Tuned": GradientBoostingClassifier(
            n_estimators=300, max_depth=3, random_state=random_state
        ),
        "SVC ST Tuned": SVC(
            kernel="rbf", C=10, probability=True, class_weight="balanced", random_state=random_state
        ),
        "LR ST Tuned": LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=random_state
        ),
        "KNN ST Tuned": KNeighborsClassifier(n_neighbors=5),
        "DT ST Tuned": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=random_state)
    }

    temporal_val_probs = {}
    temporal_test_probs = {}
    temporal_val_f1 = {}

    for model_name, model in temporal_models.items():
        start = time.time()

        model.fit(X_train, y[train_idx])

        val_prob = model.predict_proba(X_val)[:, 1]
        test_prob = model.predict_proba(X_test)[:, 1]

        threshold, val_f1 = find_best_threshold(y[val_idx], val_prob)
        test_pred = predict_with_threshold(test_prob, threshold)

        temporal_val_probs[model_name] = val_prob
        temporal_test_probs[model_name] = test_prob
        temporal_val_f1[model_name] = val_f1

        metrics = evaluate_predictions(y[test_idx], test_pred, test_prob)
        metrics.update({
            "Project": project_name,
            "Model": model_name,
            "Family": "Static + Temporal ML",
            "Feature_Set": "Static + Temporal",
            "Threshold": threshold,
            "Validation_F1": val_f1,
            "Training_Time_sec": time.time() - start
        })

        results.append(metrics)

    # ==========================================================
    # 3. Train GraphSAGE and extract graph-risk probabilities
    # ==========================================================
    X_graph_scaled = StandardScaler().fit_transform(X_st)

    x_tensor = torch.tensor(X_graph_scaled, dtype=torch.float)
    y_tensor = torch.tensor(y, dtype=torch.long)

    edge_index = build_similarity_graph(X_graph_scaled, k_neighbors=10)

    num_nodes = x_tensor.shape[0]

    train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(num_nodes, dtype=torch.bool)

    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True

    data_st = Data(
        x=x_tensor,
        edge_index=edge_index,
        y=y_tensor,
        train_mask=train_mask,
        val_mask=val_mask,
        test_mask=test_mask
    ).to(device)

    graph_model = GraphSAGEDefectPredictor(
        in_channels=data_st.x.shape[1],
        hidden_channels=64,
        out_channels=2,
        dropout=0.40
    ).to(device)

    class_counts = np.bincount(y[train_idx])
    class_weights = class_counts.sum() / (2.0 * class_counts)
    class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

    optimizer = torch.optim.AdamW(graph_model.parameters(), lr=0.003, weight_decay=1e-3)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_val_f1 = 0
    best_state = None
    best_graph_threshold = 0.50
    patience = 50
    patience_counter = 0

    start_graph = time.time()

    for epoch in range(1, 401):
        graph_model.train()
        optimizer.zero_grad()

        logits = graph_model(data_st.x, data_st.edge_index)
        loss = criterion(logits[data_st.train_mask], data_st.y[data_st.train_mask])

        loss.backward()
        optimizer.step()

        graph_model.eval()
        with torch.no_grad():
            val_logits = graph_model(data_st.x, data_st.edge_index)
            val_prob_all = F.softmax(val_logits, dim=1)[:, 1].detach().cpu().numpy()
            val_prob = val_prob_all[val_idx]

        threshold, val_f1 = find_best_threshold(y[val_idx], val_prob)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_graph_threshold = threshold
            best_state = copy.deepcopy(graph_model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    if best_state is not None:
        graph_model.load_state_dict(best_state)

    graph_model.eval()
    with torch.no_grad():
        logits = graph_model(data_st.x, data_st.edge_index)
        graph_prob_all = F.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()

    graph_val_prob = graph_prob_all[val_idx]
    graph_test_prob = graph_prob_all[test_idx]

    graph_test_pred = predict_with_threshold(graph_test_prob, best_graph_threshold)

    graph_metrics = evaluate_predictions(y[test_idx], graph_test_pred, graph_test_prob)
    graph_metrics.update({
        "Project": project_name,
        "Model": "RATE-FP GraphSAGE ST Tuned",
        "Family": "Graph Model",
        "Feature_Set": "Graph Static + Temporal",
        "Threshold": best_graph_threshold,
        "Validation_F1": best_val_f1,
        "Training_Time_sec": time.time() - start_graph
    })

    results.append(graph_metrics)

    # ==========================================================
    # 4. Graph-risk augmented features
    # ==========================================================
    graph_risk_feature = graph_prob_all.reshape(-1, 1)

    X_augmented = np.hstack([
        X_st,
        graph_risk_feature
    ])

    scaler_aug = StandardScaler()

    X_aug_train = scaler_aug.fit_transform(X_augmented[train_idx])
    X_aug_val = scaler_aug.transform(X_augmented[val_idx])
    X_aug_test = scaler_aug.transform(X_augmented[test_idx])

    graph_risk_models = {
        "RATE-FP-GR RF": RandomForestClassifier(
            n_estimators=400,
            max_depth=12,
            class_weight="balanced",
            random_state=random_state
        ),
        "RATE-FP-GR GBR": GradientBoostingClassifier(
            n_estimators=400,
            max_depth=3,
            learning_rate=0.03,
            random_state=random_state
        ),
        "RATE-FP-GR SVC": SVC(
            kernel="rbf",
            C=10,
            probability=True,
            class_weight="balanced",
            random_state=random_state
        ),
        "RATE-FP-GR LR": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=random_state
        )
    }

    gr_val_probs = {}
    gr_test_probs = {}
    gr_val_f1 = {}

    for model_name, model in graph_risk_models.items():
        start = time.time()

        model.fit(X_aug_train, y[train_idx])

        val_prob = model.predict_proba(X_aug_val)[:, 1]
        test_prob = model.predict_proba(X_aug_test)[:, 1]

        threshold, val_f1 = find_best_threshold(y[val_idx], val_prob)
        test_pred = predict_with_threshold(test_prob, threshold)

        gr_val_probs[model_name] = val_prob
        gr_test_probs[model_name] = test_prob
        gr_val_f1[model_name] = val_f1

        metrics = evaluate_predictions(y[test_idx], test_pred, test_prob)
        metrics.update({
            "Project": project_name,
            "Model": model_name,
            "Family": "Proposed RATE-FP-GR",
            "Feature_Set": "Static + Temporal + Graph Risk",
            "Threshold": threshold,
            "Validation_F1": val_f1,
            "Training_Time_sec": time.time() - start
        })

        results.append(metrics)

    # ==========================================================
    # 5. Graph-risk augmented weighted ensemble
    # ==========================================================
    selected_for_gr_ensemble = list(graph_risk_models.keys())

    raw_weights = np.array([gr_val_f1[m] for m in selected_for_gr_ensemble], dtype=float) + 1e-6
    adaptive_weights = raw_weights / raw_weights.sum()

    final_val_prob = np.zeros_like(y[val_idx], dtype=float)
    final_test_prob = np.zeros_like(y[test_idx], dtype=float)

    for model_name, weight in zip(selected_for_gr_ensemble, adaptive_weights):
        final_val_prob += weight * gr_val_probs[model_name]
        final_test_prob += weight * gr_test_probs[model_name]

    ensemble_threshold, ensemble_val_f1 = find_best_threshold(y[val_idx], final_val_prob)
    ensemble_test_pred = predict_with_threshold(final_test_prob, ensemble_threshold)

    ensemble_metrics = evaluate_predictions(y[test_idx], ensemble_test_pred, final_test_prob)
    ensemble_metrics.update({
        "Project": project_name,
        "Model": "RATE-FP-GR Adaptive Ensemble",
        "Family": "Proposed RATE-FP-GR",
        "Feature_Set": "Graph-Risk Augmented Ensemble",
        "Threshold": ensemble_threshold,
        "Validation_F1": ensemble_val_f1,
        "Training_Time_sec": np.nan
    })

    results.append(ensemble_metrics)

    result_df = pd.DataFrame(results)

    # Select final RATE-FP-GR model by validation F1 among proposed graph-risk models
    proposed_gr_df = result_df[result_df["Family"] == "Proposed RATE-FP-GR"].copy()
    selected_gr = proposed_gr_df.sort_values("Validation_F1", ascending=False).iloc[0].copy()
    selected_gr["Model"] = "RATE-FP-GR Validation-Guided Selector"
    selected_gr["Family"] = "Final Proposed RATE-FP-GR"
    result_df = pd.concat([result_df, pd.DataFrame([selected_gr])], ignore_index=True)

    result_path = RESULTS_DIR / f"{project_name}_GRAPH_RISK_AUGMENTED_RESULTS_seed_{random_state}.csv"
    result_df.to_csv(result_path, index=False)

    print(f"Completed Graph-Risk Augmented {project_name}")
    print("Selected RATE-FP-GR model:", selected_gr["Feature_Set"])
    print("Selected RATE-FP-GR F1:", selected_gr["F1"])
    print("Selected RATE-FP-GR ROC-AUC:", selected_gr["ROC_AUC"])
    print("Selected RATE-FP-GR Recall:", selected_gr["Recall"])

    return result_df

In [ ]:
graph_risk_results = []

for project_name, df in all_project_data.items():
    result_df = train_project_models_graph_risk_augmented(
        project_name,
        df,
        random_state=42
    )
    graph_risk_results.append(result_df)

graph_risk_all_df = pd.concat(graph_risk_results, ignore_index=True)

graph_risk_path = RESULTS_DIR / "GRAPH_RISK_AUGMENTED_ALL_PROJECTS_seed_42.csv"
graph_risk_all_df.to_csv(graph_risk_path, index=False)

print("Saved graph-risk augmented results to:")
print(graph_risk_path)

display(graph_risk_all_df.sort_values(["Project", "F1"], ascending=[True, False]))


Training Graph-Risk Augmented RATE-FP project: jdt
Completed Graph-Risk Augmented jdt
Selected RATE-FP-GR model: Graph-Risk Augmented Ensemble
Selected RATE-FP-GR F1: 0.7058823529411765
Selected RATE-FP-GR ROC-AUC: 0.897804283003524
Selected RATE-FP-GR Recall: 0.7741935483870968

Training Graph-Risk Augmented RATE-FP project: pde
Completed Graph-Risk Augmented pde
Selected RATE-FP-GR model: Static + Temporal + Graph Risk
Selected RATE-FP-GR F1: 0.5217391304347826
Selected RATE-FP-GR ROC-AUC: 0.8119388094446293
Selected RATE-FP-GR Recall: 0.5806451612903226

Training Graph-Risk Augmented RATE-FP project: equinox
Completed Graph-Risk Augmented equinox
Selected RATE-FP-GR model: Static + Temporal + Graph Risk
Selected RATE-FP-GR F1: 0.5853658536585366
Selected RATE-FP-GR ROC-AUC: 0.7543859649122807
Selected RATE-FP-GR Recall: 0.631578947368421

Training Graph-Risk Augmented RATE-FP project: lucene
Completed Graph-Risk Augmented lucene
Selected RATE-FP-GR model: Static + Temporal + Graph 

,Accuracy,Precision,Recall,F1,ROC_AUC,Project,Model,Family,Feature_Set,Threshold,Validation_F1,Training_Time_sec
53,0.693878,0.566667,0.894737,0.693878,0.766667,equinox,RATE-FP-GR SVC,Proposed RATE-FP-GR,Static + Temporal + Graph Risk,0.32,0.765957,0.155744
38,0.734694,0.636364,0.736842,0.682927,0.782456,equinox,GBR Static Tuned,Static-only ML,Static,0.40,0.857143,0.582701
41,0.714286,0.600000,0.789474,0.681818,0.728070,equinox,SVC Static Tuned,Static-only ML,Static,0.37,0.790698,0.149883
39,0.673469,0.555556,0.789474,0.652174,0.780702,equinox,RF Static Tuned,Static-only ML,Static,0.44,0.837209,0.483287
46,0.693878,0.583333,0.736842,0.651163,0.770175,equinox,SVC ST Tuned,Static + Temporal ML,Static + Temporal,0.44,0.761905,0.165808
...,...,...,...,...,...,...,...,...,...,...,...,...
22,0.586667,0.223214,0.806452,0.349650,0.768540,pde,SVC Static Tuned,Static-only ML,Static,0.13,0.317241,0.387813
19,0.746667,0.267857,0.483871,0.344828,0.772531,pde,GBR Static Tuned,Static-only ML,Static,0.14,0.449438,1.470119
27,0.773333,0.272727,0.387097,0.320000,0.730961,pde,SVC ST Tuned,Static + Temporal ML,Static + Temporal,0.19,0.347826,0.707074
29,0.635556,0.213483,0.612903,0.316667,0.650482,pde,KNN ST Tuned,Static + Temporal ML,Static + Temporal,0.10,0.375940,0.136285


In [ ]:
summary_rows = []

for project in graph_risk_all_df["Project"].unique():
    project_df = graph_risk_all_df[graph_risk_all_df["Project"] == project].copy()

    static_best = (
        project_df[project_df["Family"] == "Static-only ML"]
        .sort_values("F1", ascending=False)
        .iloc[0]
    )

    temporal_best = (
        project_df[project_df["Family"] == "Static + Temporal ML"]
        .sort_values("F1", ascending=False)
        .iloc[0]
    )

    graph_model = (
        project_df[project_df["Model"] == "RATE-FP GraphSAGE ST Tuned"]
        .iloc[0]
    )

    proposed_final = (
        project_df[project_df["Model"] == "RATE-FP-GR Validation-Guided Selector"]
        .iloc[0]
    )

    summary_rows.append({
        "Project": project,

        "Best_Static_Model": static_best["Model"],
        "Best_Static_F1": static_best["F1"],
        "Best_Static_ROC_AUC": static_best["ROC_AUC"],
        "Best_Static_Recall": static_best["Recall"],

        "Best_Temporal_Model": temporal_best["Model"],
        "Best_Temporal_F1": temporal_best["F1"],
        "Best_Temporal_ROC_AUC": temporal_best["ROC_AUC"],
        "Best_Temporal_Recall": temporal_best["Recall"],

        "GraphSAGE_F1": graph_model["F1"],
        "GraphSAGE_ROC_AUC": graph_model["ROC_AUC"],
        "GraphSAGE_Recall": graph_model["Recall"],

        "RATE_FP_GR_Model": proposed_final["Feature_Set"],
        "RATE_FP_GR_F1": proposed_final["F1"],
        "RATE_FP_GR_ROC_AUC": proposed_final["ROC_AUC"],
        "RATE_FP_GR_Accuracy": proposed_final["Accuracy"],
        "RATE_FP_GR_Precision": proposed_final["Precision"],
        "RATE_FP_GR_Recall": proposed_final["Recall"],

        "F1_Gain_vs_Static": proposed_final["F1"] - static_best["F1"],
        "F1_Gain_vs_Temporal": proposed_final["F1"] - temporal_best["F1"],
        "ROC_Gain_vs_Static": proposed_final["ROC_AUC"] - static_best["ROC_AUC"],
        "ROC_Gain_vs_Temporal": proposed_final["ROC_AUC"] - temporal_best["ROC_AUC"],
    })

graph_risk_summary_df = pd.DataFrame(summary_rows)

avg_row = {
    "Project": "Average",

    "Best_Static_Model": "-",
    "Best_Static_F1": graph_risk_summary_df["Best_Static_F1"].mean(),
    "Best_Static_ROC_AUC": graph_risk_summary_df["Best_Static_ROC_AUC"].mean(),
    "Best_Static_Recall": graph_risk_summary_df["Best_Static_Recall"].mean(),

    "Best_Temporal_Model": "-",
    "Best_Temporal_F1": graph_risk_summary_df["Best_Temporal_F1"].mean(),
    "Best_Temporal_ROC_AUC": graph_risk_summary_df["Best_Temporal_ROC_AUC"].mean(),
    "Best_Temporal_Recall": graph_risk_summary_df["Best_Temporal_Recall"].mean(),

    "GraphSAGE_F1": graph_risk_summary_df["GraphSAGE_F1"].mean(),
    "GraphSAGE_ROC_AUC": graph_risk_summary_df["GraphSAGE_ROC_AUC"].mean(),
    "GraphSAGE_Recall": graph_risk_summary_df["GraphSAGE_Recall"].mean(),

    "RATE_FP_GR_Model": "-",
    "RATE_FP_GR_F1": graph_risk_summary_df["RATE_FP_GR_F1"].mean(),
    "RATE_FP_GR_ROC_AUC": graph_risk_summary_df["RATE_FP_GR_ROC_AUC"].mean(),
    "RATE_FP_GR_Accuracy": graph_risk_summary_df["RATE_FP_GR_Accuracy"].mean(),
    "RATE_FP_GR_Precision": graph_risk_summary_df["RATE_FP_GR_Precision"].mean(),
    "RATE_FP_GR_Recall": graph_risk_summary_df["RATE_FP_GR_Recall"].mean(),

    "F1_Gain_vs_Static": graph_risk_summary_df["F1_Gain_vs_Static"].mean(),
    "F1_Gain_vs_Temporal": graph_risk_summary_df["F1_Gain_vs_Temporal"].mean(),
    "ROC_Gain_vs_Static": graph_risk_summary_df["ROC_Gain_vs_Static"].mean(),
    "ROC_Gain_vs_Temporal": graph_risk_summary_df["ROC_Gain_vs_Temporal"].mean(),
}

graph_risk_summary_df = pd.concat(
    [graph_risk_summary_df, pd.DataFrame([avg_row])],
    ignore_index=True
)

display(graph_risk_summary_df)

graph_risk_summary_path = RESULTS_DIR / "GRAPH_RISK_AUGMENTED_SUMMARY_seed_42.csv"
graph_risk_summary_df.to_csv(graph_risk_summary_path, index=False)

print("Saved graph-risk augmented summary to:")
print(graph_risk_summary_path)

,Project,Best_Static_Model,Best_Static_F1,Best_Static_ROC_AUC,Best_Static_Recall,Best_Temporal_Model,Best_Temporal_F1,Best_Temporal_ROC_AUC,Best_Temporal_Recall,GraphSAGE_F1,...,RATE_FP_GR_Model,RATE_FP_GR_F1,RATE_FP_GR_ROC_AUC,RATE_FP_GR_Accuracy,RATE_FP_GR_Precision,RATE_FP_GR_Recall,F1_Gain_vs_Static,F1_Gain_vs_Temporal,ROC_Gain_vs_Static,ROC_Gain_vs_Temporal
0,jdt,GBR Static Tuned,0.686567,0.888588,0.741935,RF ST Tuned,0.704225,0.898346,0.806452,0.655172,...,Graph-Risk Augmented Ensemble,0.705882,0.897804,0.866667,0.648649,0.774194,0.019315,0.001657,0.009217,-0.000542
1,pde,RF Static Tuned,0.477612,0.803126,0.516129,GBR ST Tuned,0.441176,0.808447,0.483871,0.400000,...,Static + Temporal + Graph Risk,0.521739,0.811939,0.853333,0.473684,0.580645,0.044127,0.080563,0.008813,0.003492
2,equinox,GBR Static Tuned,0.682927,0.782456,0.736842,SVC ST Tuned,0.651163,0.770175,0.736842,0.604651,...,Static + Temporal + Graph Risk,0.585366,0.754386,0.653061,0.545455,0.631579,-0.097561,-0.065797,-0.028070,-0.015789
3,lucene,RF Static Tuned,0.250000,0.700585,0.333333,GBR ST Tuned,0.285714,0.705263,0.333333,0.307692,...,Static + Temporal + Graph Risk,0.285714,0.748538,0.759615,0.192308,0.555556,0.035714,0.000000,0.047953,0.043275
4,mylyn,RF Static Tuned,0.536585,0.835836,0.594595,RF ST Tuned,0.495868,0.847181,0.810811,0.402985,...,Static + Temporal + Graph Risk,0.369427,0.825270,0.646429,0.241667,0.783784,-0.167159,-0.126441,-0.010566,-0.021911
5,Average,-,0.526738,0.802118,0.584567,-,0.515629,0.805883,0.634262,0.474100,...,-,0.493626,0.807587,0.755821,0.420352,0.665151,-0.033113,-0.022004,0.005469,0.001705


Saved graph-risk augmented summary to:
/content/drive/MyDrive/RATE_FP_Project/results/GRAPH_RISK_AUGMENTED_SUMMARY_seed_42.csv


In [ ]:
!pip install -q xgboost imbalanced-learn

In [ ]:
from xgboost import XGBClassifier
from imblearn.ensemble import BalancedRandomForestClassifier, EasyEnsembleClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier

In [ ]:
def train_project_models_imbalance_aware_gr(project_name, rate_fp_df, random_state=42):
    print("\n" + "="*120)
    print(f"Training Imbalance-Aware RATE-FP-GR project: {project_name}")
    print("="*120)

    rate_fp_df = rate_fp_df.copy()

    for col in rate_fp_df.columns:
        if col != "classname":
            rate_fp_df[col] = pd.to_numeric(rate_fp_df[col], errors="coerce")

    rate_fp_df = rate_fp_df.fillna(0)
    rate_fp_df["is_defective"] = (rate_fp_df["bugs"] > 0).astype(int)

    target_cols = [
        "bugs", "nonTrivialBugs", "majorBugs",
        "criticalBugs", "highPriorityBugs", "is_defective"
    ]

    static_temporal_feature_cols = [
        col for col in rate_fp_df.columns
        if col not in ["classname"] + target_cols
    ]

    y = rate_fp_df["is_defective"].values
    indices_all = np.arange(len(rate_fp_df))

    train_idx, temp_idx, y_train, y_temp = train_test_split(
        indices_all,
        y,
        test_size=0.30,
        random_state=random_state,
        stratify=y
    )

    val_idx, test_idx, y_val, y_test = train_test_split(
        temp_idx,
        y_temp,
        test_size=0.50,
        random_state=random_state,
        stratify=y_temp
    )

    X_st = rate_fp_df[static_temporal_feature_cols].values

    # =====================================================
    # 1. Train GraphSAGE to generate graph-risk score
    # =====================================================
    X_graph_scaled = StandardScaler().fit_transform(X_st)

    x_tensor = torch.tensor(X_graph_scaled, dtype=torch.float)
    y_tensor = torch.tensor(y, dtype=torch.long)

    edge_index = build_similarity_graph(X_graph_scaled, k_neighbors=10)

    num_nodes = x_tensor.shape[0]

    train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(num_nodes, dtype=torch.bool)

    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True

    data_st = Data(
        x=x_tensor,
        edge_index=edge_index,
        y=y_tensor,
        train_mask=train_mask,
        val_mask=val_mask,
        test_mask=test_mask
    ).to(device)

    graph_model = GraphSAGEDefectPredictor(
        in_channels=data_st.x.shape[1],
        hidden_channels=64,
        out_channels=2,
        dropout=0.40
    ).to(device)

    class_counts = np.bincount(y[train_idx])
    class_weights = class_counts.sum() / (2.0 * class_counts)
    class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

    optimizer = torch.optim.AdamW(graph_model.parameters(), lr=0.003, weight_decay=1e-3)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_val_f1 = 0
    best_state = None
    patience = 50
    patience_counter = 0

    for epoch in range(1, 401):
        graph_model.train()
        optimizer.zero_grad()

        logits = graph_model(data_st.x, data_st.edge_index)
        loss = criterion(logits[data_st.train_mask], data_st.y[data_st.train_mask])

        loss.backward()
        optimizer.step()

        graph_model.eval()
        with torch.no_grad():
            val_logits = graph_model(data_st.x, data_st.edge_index)
            val_prob_all = F.softmax(val_logits, dim=1)[:, 1].detach().cpu().numpy()
            val_prob = val_prob_all[val_idx]

        threshold, val_f1 = find_best_threshold(y[val_idx], val_prob)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = copy.deepcopy(graph_model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    if best_state is not None:
        graph_model.load_state_dict(best_state)

    graph_model.eval()
    with torch.no_grad():
        logits = graph_model(data_st.x, data_st.edge_index)
        graph_prob_all = F.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()

    graph_risk_feature = graph_prob_all.reshape(-1, 1)

    # Final augmented feature matrix
    X_augmented = np.hstack([X_st, graph_risk_feature])

    scaler_aug = StandardScaler()
    X_train = scaler_aug.fit_transform(X_augmented[train_idx])
    X_val = scaler_aug.transform(X_augmented[val_idx])
    X_test = scaler_aug.transform(X_augmented[test_idx])

    y_train = y[train_idx]
    y_val = y[val_idx]
    y_test = y[test_idx]

    neg_count, pos_count = np.bincount(y_train)
    scale_pos_weight = neg_count / max(pos_count, 1)

    # =====================================================
    # 2. Imbalance-aware models
    # =====================================================
    imbalance_models = {
        "RATE-FP-GR XGBoost": XGBClassifier(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.03,
            subsample=0.85,
            colsample_bytree=0.85,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=random_state
        ),

        "RATE-FP-GR Balanced RF": BalancedRandomForestClassifier(
            n_estimators=400,
            max_depth=12,
            sampling_strategy="auto",
            replacement=True,
            random_state=random_state
        ),

        "RATE-FP-GR Easy Ensemble": EasyEnsembleClassifier(
            n_estimators=30,
            random_state=random_state
        ),

        "RATE-FP-GR Extra Trees": ExtraTreesClassifier(
            n_estimators=400,
            max_depth=12,
            class_weight="balanced",
            random_state=random_state
        ),

        "RATE-FP-GR HistGB": HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.03,
            max_leaf_nodes=31,
            random_state=random_state
        )
    }

    results = []
    val_probs = {}
    test_probs = {}
    val_f1_scores = {}

    for model_name, model in imbalance_models.items():
        start = time.time()

        model.fit(X_train, y_train)

        val_prob = model.predict_proba(X_val)[:, 1]
        test_prob = model.predict_proba(X_test)[:, 1]

        threshold, val_f1 = find_best_threshold(y_val, val_prob)
        test_pred = predict_with_threshold(test_prob, threshold)

        metrics = evaluate_predictions(y_test, test_pred, test_prob)

        metrics.update({
            "Project": project_name,
            "Model": model_name,
            "Family": "Imbalance-Aware RATE-FP-GR",
            "Feature_Set": "Static + Temporal + Graph Risk",
            "Threshold": threshold,
            "Validation_F1": val_f1,
            "Training_Time_sec": time.time() - start
        })

        results.append(metrics)

        val_probs[model_name] = val_prob
        test_probs[model_name] = test_prob
        val_f1_scores[model_name] = val_f1

    # =====================================================
    # 3. Weighted probability ensemble
    # =====================================================
    selected_models = list(imbalance_models.keys())

    raw_weights = np.array([val_f1_scores[m] for m in selected_models]) + 1e-6
    weights = raw_weights / raw_weights.sum()

    final_val_prob = np.zeros_like(y_val, dtype=float)
    final_test_prob = np.zeros_like(y_test, dtype=float)

    for model_name, weight in zip(selected_models, weights):
        final_val_prob += weight * val_probs[model_name]
        final_test_prob += weight * test_probs[model_name]

    ensemble_threshold, ensemble_val_f1 = find_best_threshold(y_val, final_val_prob)
    ensemble_test_pred = predict_with_threshold(final_test_prob, ensemble_threshold)

    ensemble_metrics = evaluate_predictions(y_test, ensemble_test_pred, final_test_prob)
    ensemble_metrics.update({
        "Project": project_name,
        "Model": "RATE-FP-GR Imbalance-Aware Ensemble",
        "Family": "Final Proposed RATE-FP-GR",
        "Feature_Set": "Imbalance-Aware Graph-Risk Ensemble",
        "Threshold": ensemble_threshold,
        "Validation_F1": ensemble_val_f1,
        "Training_Time_sec": np.nan
    })

    results.append(ensemble_metrics)

    result_df = pd.DataFrame(results)

    selected_final = result_df.sort_values("Validation_F1", ascending=False).iloc[0].copy()
    selected_final["Model"] = "RATE-FP-GR Imbalance-Aware Selector"
    selected_final["Family"] = "Final Proposed RATE-FP-GR"

    result_df = pd.concat([result_df, pd.DataFrame([selected_final])], ignore_index=True)

    print(f"Completed Imbalance-Aware RATE-FP-GR {project_name}")
    print("Selected model:", selected_final["Feature_Set"])
    print("Selected F1:", selected_final["F1"])
    print("Selected ROC-AUC:", selected_final["ROC_AUC"])
    print("Selected Recall:", selected_final["Recall"])

    return result_df

In [ ]:
imbalance_results = []

for project_name, df in all_project_data.items():
    result_df = train_project_models_imbalance_aware_gr(
        project_name,
        df,
        random_state=42
    )
    imbalance_results.append(result_df)

imbalance_all_df = pd.concat(imbalance_results, ignore_index=True)

imbalance_path = RESULTS_DIR / "IMBALANCE_AWARE_RATE_FP_GR_seed_42.csv"
imbalance_all_df.to_csv(imbalance_path, index=False)

print("Saved imbalance-aware RATE-FP-GR results to:")
print(imbalance_path)

display(imbalance_all_df.sort_values(["Project", "F1"], ascending=[True, False]))


Training Imbalance-Aware RATE-FP-GR project: jdt
Completed Imbalance-Aware RATE-FP-GR jdt
Selected model: Static + Temporal + Graph Risk
Selected F1: 0.7741935483870968
Selected ROC-AUC: 0.9181349959338574
Selected Recall: 0.7741935483870968

Training Imbalance-Aware RATE-FP-GR project: pde
Completed Imbalance-Aware RATE-FP-GR pde
Selected model: Static + Temporal + Graph Risk
Selected F1: 0.4838709677419355
Selected ROC-AUC: 0.8084469571000997
Selected Recall: 0.4838709677419355

Training Imbalance-Aware RATE-FP-GR project: equinox
Completed Imbalance-Aware RATE-FP-GR equinox
Selected model: Static + Temporal + Graph Risk
Selected F1: 0.5263157894736842
Selected ROC-AUC: 0.757017543859649
Selected Recall: 0.5263157894736842

Training Imbalance-Aware RATE-FP-GR project: lucene
Completed Imbalance-Aware RATE-FP-GR lucene
Selected model: Static + Temporal + Graph Risk
Selected F1: 0.125
Selected ROC-AUC: 0.7181286549707602
Selected Recall: 0.1111111111111111

Training Imbalance-Aware RA

,Accuracy,Precision,Recall,F1,ROC_AUC,Project,Model,Family,Feature_Set,Threshold,Validation_F1,Training_Time_sec
14,0.653061,0.545455,0.631579,0.585366,0.580702,equinox,RATE-FP-GR XGBoost,Imbalance-Aware RATE-FP-GR,Static + Temporal + Graph Risk,0.19,0.809524,0.306183
16,0.653061,0.545455,0.631579,0.585366,0.635088,equinox,RATE-FP-GR Easy Ensemble,Imbalance-Aware RATE-FP-GR,Static + Temporal + Graph Risk,0.48,0.809524,6.032857
17,0.653061,0.545455,0.631579,0.585366,0.731579,equinox,RATE-FP-GR Extra Trees,Imbalance-Aware RATE-FP-GR,Static + Temporal + Graph Risk,0.51,0.800000,0.735640
19,0.653061,0.545455,0.631579,0.585366,0.735088,equinox,RATE-FP-GR Imbalance-Aware Ensemble,Final Proposed RATE-FP-GR,Imbalance-Aware Graph-Risk Ensemble,0.81,0.820513,NaN
18,0.632653,0.521739,0.631579,0.571429,0.640351,equinox,RATE-FP-GR HistGB,Imbalance-Aware RATE-FP-GR,Static + Temporal + Graph Risk,0.13,0.809524,0.668925
15,0.632653,0.526316,0.526316,0.526316,0.757018,equinox,RATE-FP-GR Balanced RF,Imbalance-Aware RATE-FP-GR,Static + Temporal + Graph Risk,0.74,0.842105,1.309672
20,0.632653,0.526316,0.526316,0.526316,0.757018,equinox,RATE-FP-GR Imbalance-Aware Selector,Final Proposed RATE-FP-GR,Static + Temporal + Graph Risk,0.74,0.842105,1.309672
3,0.906667,0.774194,0.774194,0.774194,0.918135,jdt,RATE-FP-GR Extra Trees,Imbalance-Aware RATE-FP-GR,Static + Temporal + Graph Risk,0.51,0.698413,1.336842
6,0.906667,0.774194,0.774194,0.774194,0.918135,jdt,RATE-FP-GR Imbalance-Aware Selector,Final Proposed RATE-FP-GR,Static + Temporal + Graph Risk,0.51,0.698413,1.336842
1,0.886667,0.705882,0.774194,0.738462,0.910680,jdt,RATE-FP-GR Balanced RF,Imbalance-Aware RATE-FP-GR,Static + Temporal + Graph Risk,0.70,0.666667,1.702115


In [ ]:
summary_rows = []

# Use previous graph_risk_all_df if available
# If not, load it:
graph_risk_path = RESULTS_DIR / "GRAPH_RISK_AUGMENTED_ALL_PROJECTS_seed_42.csv"
graph_risk_all_df = pd.read_csv(graph_risk_path)

for project in imbalance_all_df["Project"].unique():
    imba_df = imbalance_all_df[imbalance_all_df["Project"] == project].copy()
    old_df = graph_risk_all_df[graph_risk_all_df["Project"] == project].copy()

    static_best = (
        old_df[old_df["Family"] == "Static-only ML"]
        .sort_values("F1", ascending=False)
        .iloc[0]
    )

    temporal_best = (
        old_df[old_df["Family"] == "Static + Temporal ML"]
        .sort_values("F1", ascending=False)
        .iloc[0]
    )

    graph_risk_old = (
        old_df[old_df["Model"] == "RATE-FP-GR Validation-Guided Selector"]
        .iloc[0]
    )

    imba_final = (
        imba_df[imba_df["Model"] == "RATE-FP-GR Imbalance-Aware Selector"]
        .iloc[0]
    )

    summary_rows.append({
        "Project": project,

        "Best_Static_F1": static_best["F1"],
        "Best_Static_ROC_AUC": static_best["ROC_AUC"],
        "Best_Static_Recall": static_best["Recall"],

        "Best_Temporal_F1": temporal_best["F1"],
        "Best_Temporal_ROC_AUC": temporal_best["ROC_AUC"],
        "Best_Temporal_Recall": temporal_best["Recall"],

        "Old_GRAPH_RISK_F1": graph_risk_old["F1"],
        "Old_GRAPH_RISK_ROC_AUC": graph_risk_old["ROC_AUC"],
        "Old_GRAPH_RISK_Recall": graph_risk_old["Recall"],

        "Imbalance_RATE_FP_GR_F1": imba_final["F1"],
        "Imbalance_RATE_FP_GR_ROC_AUC": imba_final["ROC_AUC"],
        "Imbalance_RATE_FP_GR_Accuracy": imba_final["Accuracy"],
        "Imbalance_RATE_FP_GR_Precision": imba_final["Precision"],
        "Imbalance_RATE_FP_GR_Recall": imba_final["Recall"],

        "F1_Gain_vs_Static": imba_final["F1"] - static_best["F1"],
        "F1_Gain_vs_Temporal": imba_final["F1"] - temporal_best["F1"],
        "F1_Gain_vs_Old_GR": imba_final["F1"] - graph_risk_old["F1"],

        "ROC_Gain_vs_Static": imba_final["ROC_AUC"] - static_best["ROC_AUC"],
        "ROC_Gain_vs_Temporal": imba_final["ROC_AUC"] - temporal_best["ROC_AUC"],
        "ROC_Gain_vs_Old_GR": imba_final["ROC_AUC"] - graph_risk_old["ROC_AUC"],
    })

imbalance_summary_df = pd.DataFrame(summary_rows)

avg_row = {
    "Project": "Average",

    "Best_Static_F1": imbalance_summary_df["Best_Static_F1"].mean(),
    "Best_Static_ROC_AUC": imbalance_summary_df["Best_Static_ROC_AUC"].mean(),
    "Best_Static_Recall": imbalance_summary_df["Best_Static_Recall"].mean(),

    "Best_Temporal_F1": imbalance_summary_df["Best_Temporal_F1"].mean(),
    "Best_Temporal_ROC_AUC": imbalance_summary_df["Best_Temporal_ROC_AUC"].mean(),
    "Best_Temporal_Recall": imbalance_summary_df["Best_Temporal_Recall"].mean(),

    "Old_GRAPH_RISK_F1": imbalance_summary_df["Old_GRAPH_RISK_F1"].mean(),
    "Old_GRAPH_RISK_ROC_AUC": imbalance_summary_df["Old_GRAPH_RISK_ROC_AUC"].mean(),
    "Old_GRAPH_RISK_Recall": imbalance_summary_df["Old_GRAPH_RISK_Recall"].mean(),

    "Imbalance_RATE_FP_GR_F1": imbalance_summary_df["Imbalance_RATE_FP_GR_F1"].mean(),
    "Imbalance_RATE_FP_GR_ROC_AUC": imbalance_summary_df["Imbalance_RATE_FP_GR_ROC_AUC"].mean(),
    "Imbalance_RATE_FP_GR_Accuracy": imbalance_summary_df["Imbalance_RATE_FP_GR_Accuracy"].mean(),
    "Imbalance_RATE_FP_GR_Precision": imbalance_summary_df["Imbalance_RATE_FP_GR_Precision"].mean(),
    "Imbalance_RATE_FP_GR_Recall": imbalance_summary_df["Imbalance_RATE_FP_GR_Recall"].mean(),

    "F1_Gain_vs_Static": imbalance_summary_df["F1_Gain_vs_Static"].mean(),
    "F1_Gain_vs_Temporal": imbalance_summary_df["F1_Gain_vs_Temporal"].mean(),
    "F1_Gain_vs_Old_GR": imbalance_summary_df["F1_Gain_vs_Old_GR"].mean(),

    "ROC_Gain_vs_Static": imbalance_summary_df["ROC_Gain_vs_Static"].mean(),
    "ROC_Gain_vs_Temporal": imbalance_summary_df["ROC_Gain_vs_Temporal"].mean(),
    "ROC_Gain_vs_Old_GR": imbalance_summary_df["ROC_Gain_vs_Old_GR"].mean()
}

imbalance_summary_df = pd.concat(
    [imbalance_summary_df, pd.DataFrame([avg_row])],
    ignore_index=True
)

display(imbalance_summary_df)

imbalance_summary_path = RESULTS_DIR / "IMBALANCE_AWARE_RATE_FP_GR_SUMMARY_seed_42.csv"
imbalance_summary_df.to_csv(imbalance_summary_path, index=False)

print("Saved imbalance-aware RATE-FP-GR summary to:")
print(imbalance_summary_path)

,Project,Best_Static_F1,Best_Static_ROC_AUC,Best_Static_Recall,Best_Temporal_F1,Best_Temporal_ROC_AUC,Best_Temporal_Recall,Old_GRAPH_RISK_F1,Old_GRAPH_RISK_ROC_AUC,Old_GRAPH_RISK_Recall,...,Imbalance_RATE_FP_GR_ROC_AUC,Imbalance_RATE_FP_GR_Accuracy,Imbalance_RATE_FP_GR_Precision,Imbalance_RATE_FP_GR_Recall,F1_Gain_vs_Static,F1_Gain_vs_Temporal,F1_Gain_vs_Old_GR,ROC_Gain_vs_Static,ROC_Gain_vs_Temporal,ROC_Gain_vs_Old_GR
0,jdt,0.686567,0.888588,0.741935,0.704225,0.898346,0.806452,0.705882,0.897804,0.774194,...,0.918135,0.906667,0.774194,0.774194,0.087626,0.069968,0.068311,0.029547,1.978856e-02,0.020331
1,pde,0.477612,0.803126,0.516129,0.441176,0.808447,0.483871,0.521739,0.811939,0.580645,...,0.808447,0.857778,0.483871,0.483871,0.006259,0.042694,-0.037868,0.005321,-1.110223e-16,-0.003492
2,equinox,0.682927,0.782456,0.736842,0.651163,0.770175,0.736842,0.585366,0.754386,0.631579,...,0.757018,0.632653,0.526316,0.526316,-0.156611,-0.124847,-0.059050,-0.025439,-1.315789e-02,0.002632
3,lucene,0.250000,0.700585,0.333333,0.285714,0.705263,0.333333,0.285714,0.748538,0.555556,...,0.718129,0.865385,0.142857,0.111111,-0.125000,-0.160714,-0.160714,0.017544,1.286550e-02,-0.030409
4,mylyn,0.536585,0.835836,0.594595,0.495868,0.847181,0.810811,0.369427,0.825270,0.783784,...,0.836614,0.835714,0.421053,0.648649,-0.025947,0.014771,0.141212,0.000779,-1.056612e-02,0.011345
5,Average,0.526738,0.802118,0.584567,0.515629,0.805883,0.634262,0.493626,0.807587,0.665151,...,0.807669,0.819639,0.469658,0.508828,-0.042735,-0.031626,-0.009622,0.005550,1.786008e-03,0.000081


Saved imbalance-aware RATE-FP-GR summary to:
/content/drive/MyDrive/RATE_FP_Project/results/IMBALANCE_AWARE_RATE_FP_GR_SUMMARY_seed_42.csv


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/RATE_FP_Project")
RESULTS_DIR = BASE_DIR / "results"

imbalance_path = RESULTS_DIR / "IMBALANCE_AWARE_RATE_FP_GR_seed_42.csv"
imbalance_all_df = pd.read_csv(imbalance_path)

# We select only real candidate models, not previous selector rows
candidate_models = imbalance_all_df[
    imbalance_all_df["Model"] != "RATE-FP-GR Imbalance-Aware Selector"
].copy()

# Stability-aware validation score
# Validation_F1 is primary; ROC-AUC and recall stabilize selection on imbalanced data
candidate_models["Stability_Selection_Score"] = (
    candidate_models["Validation_F1"]
    + 0.25 * candidate_models["ROC_AUC"]
    + 0.10 * candidate_models["Recall"]
)

stable_rows = []

for project in candidate_models["Project"].unique():
    project_df = candidate_models[candidate_models["Project"] == project].copy()

    selected = project_df.sort_values(
        "Stability_Selection_Score",
        ascending=False
    ).iloc[0]

    stable_rows.append({
        "Project": project,
        "Selected_Model": selected["Model"],
        "Selected_Family": selected["Family"],
        "Feature_Set": selected["Feature_Set"],
        "Validation_F1": selected["Validation_F1"],
        "Selection_Score": selected["Stability_Selection_Score"],
        "Accuracy": selected["Accuracy"],
        "Precision": selected["Precision"],
        "Recall": selected["Recall"],
        "F1": selected["F1"],
        "ROC_AUC": selected["ROC_AUC"],
        "Threshold": selected["Threshold"]
    })

stable_selector_df = pd.DataFrame(stable_rows)

avg_row = {
    "Project": "Average",
    "Selected_Model": "-",
    "Selected_Family": "-",
    "Feature_Set": "-",
    "Validation_F1": stable_selector_df["Validation_F1"].mean(),
    "Selection_Score": stable_selector_df["Selection_Score"].mean(),
    "Accuracy": stable_selector_df["Accuracy"].mean(),
    "Precision": stable_selector_df["Precision"].mean(),
    "Recall": stable_selector_df["Recall"].mean(),
    "F1": stable_selector_df["F1"].mean(),
    "ROC_AUC": stable_selector_df["ROC_AUC"].mean(),
    "Threshold": stable_selector_df["Threshold"].mean()
}

stable_selector_df = pd.concat(
    [stable_selector_df, pd.DataFrame([avg_row])],
    ignore_index=True
)

display(stable_selector_df)

stable_selector_path = RESULTS_DIR / "STABILITY_REGULARIZED_RATE_FP_GR_SELECTOR_seed_42.csv"
stable_selector_df.to_csv(stable_selector_path, index=False)

print("Saved stability-regularized selector result to:")
print(stable_selector_path)

,Project,Selected_Model,Selected_Family,Feature_Set,Validation_F1,Selection_Score,Accuracy,Precision,Recall,F1,ROC_AUC,Threshold
0,jdt,RATE-FP-GR Extra Trees,Imbalance-Aware RATE-FP-GR,Static + Temporal + Graph Risk,0.698413,1.005366,0.906667,0.774194,0.774194,0.774194,0.918135,0.510
1,pde,RATE-FP-GR Extra Trees,Imbalance-Aware RATE-FP-GR,Static + Temporal + Graph Risk,0.583333,0.833832,0.857778,0.483871,0.483871,0.483871,0.808447,0.410
2,equinox,RATE-FP-GR Balanced RF,Imbalance-Aware RATE-FP-GR,Static + Temporal + Graph Risk,0.842105,1.083991,0.632653,0.526316,0.526316,0.526316,0.757018,0.740
3,lucene,RATE-FP-GR Easy Ensemble,Imbalance-Aware RATE-FP-GR,Static + Temporal + Graph Risk,0.500000,0.690643,0.865385,0.142857,0.111111,0.125000,0.718129,0.600
4,mylyn,RATE-FP-GR Balanced RF,Imbalance-Aware RATE-FP-GR,Static + Temporal + Graph Risk,0.483516,0.757535,0.835714,0.421053,0.648649,0.510638,0.836614,0.670
5,Average,-,-,-,0.621474,0.874273,0.819639,0.469658,0.508828,0.484004,0.807669,0.586


Saved stability-regularized selector result to:
/content/drive/MyDrive/RATE_FP_Project/results/STABILITY_REGULARIZED_RATE_FP_GR_SELECTOR_seed_42.csv


In [ ]:
# Load previous summaries if available
graph_risk_summary_path = RESULTS_DIR / "GRAPH_RISK_AUGMENTED_SUMMARY_seed_42.csv"
imbalance_summary_path = RESULTS_DIR / "IMBALANCE_AWARE_RATE_FP_GR_SUMMARY_seed_42.csv"

comparison_rows = []

# Static and temporal baselines from graph-risk summary
graph_summary = pd.read_csv(graph_risk_summary_path)

for project in graph_summary[graph_summary["Project"] != "Average"]["Project"]:
    row_old = graph_summary[graph_summary["Project"] == project].iloc[0]
    row_stable = stable_selector_df[stable_selector_df["Project"] == project].iloc[0]

    comparison_rows.append({
        "Project": project,

        "Best_Static_F1": row_old["Best_Static_F1"],
        "Best_Static_ROC_AUC": row_old["Best_Static_ROC_AUC"],

        "Best_Temporal_F1": row_old["Best_Temporal_F1"],
        "Best_Temporal_ROC_AUC": row_old["Best_Temporal_ROC_AUC"],

        "Old_Graph_Risk_F1": row_old["RATE_FP_GR_F1"],
        "Old_Graph_Risk_ROC_AUC": row_old["RATE_FP_GR_ROC_AUC"],

        "Stable_RATE_FP_GR_Model": row_stable["Selected_Model"],
        "Stable_RATE_FP_GR_F1": row_stable["F1"],
        "Stable_RATE_FP_GR_ROC_AUC": row_stable["ROC_AUC"],
        "Stable_RATE_FP_GR_Accuracy": row_stable["Accuracy"],
        "Stable_RATE_FP_GR_Precision": row_stable["Precision"],
        "Stable_RATE_FP_GR_Recall": row_stable["Recall"],

        "F1_Gain_vs_Static": row_stable["F1"] - row_old["Best_Static_F1"],
        "F1_Gain_vs_Temporal": row_stable["F1"] - row_old["Best_Temporal_F1"],
        "F1_Gain_vs_Old_GR": row_stable["F1"] - row_old["RATE_FP_GR_F1"],

        "ROC_Gain_vs_Static": row_stable["ROC_AUC"] - row_old["Best_Static_ROC_AUC"],
        "ROC_Gain_vs_Temporal": row_stable["ROC_AUC"] - row_old["Best_Temporal_ROC_AUC"],
        "ROC_Gain_vs_Old_GR": row_stable["ROC_AUC"] - row_old["RATE_FP_GR_ROC_AUC"]
    })

final_stable_comparison_df = pd.DataFrame(comparison_rows)

avg_row = {
    "Project": "Average",

    "Best_Static_F1": final_stable_comparison_df["Best_Static_F1"].mean(),
    "Best_Static_ROC_AUC": final_stable_comparison_df["Best_Static_ROC_AUC"].mean(),

    "Best_Temporal_F1": final_stable_comparison_df["Best_Temporal_F1"].mean(),
    "Best_Temporal_ROC_AUC": final_stable_comparison_df["Best_Temporal_ROC_AUC"].mean(),

    "Old_Graph_Risk_F1": final_stable_comparison_df["Old_Graph_Risk_F1"].mean(),
    "Old_Graph_Risk_ROC_AUC": final_stable_comparison_df["Old_Graph_Risk_ROC_AUC"].mean(),

    "Stable_RATE_FP_GR_Model": "-",
    "Stable_RATE_FP_GR_F1": final_stable_comparison_df["Stable_RATE_FP_GR_F1"].mean(),
    "Stable_RATE_FP_GR_ROC_AUC": final_stable_comparison_df["Stable_RATE_FP_GR_ROC_AUC"].mean(),
    "Stable_RATE_FP_GR_Accuracy": final_stable_comparison_df["Stable_RATE_FP_GR_Accuracy"].mean(),
    "Stable_RATE_FP_GR_Precision": final_stable_comparison_df["Stable_RATE_FP_GR_Precision"].mean(),
    "Stable_RATE_FP_GR_Recall": final_stable_comparison_df["Stable_RATE_FP_GR_Recall"].mean(),

    "F1_Gain_vs_Static": final_stable_comparison_df["F1_Gain_vs_Static"].mean(),
    "F1_Gain_vs_Temporal": final_stable_comparison_df["F1_Gain_vs_Temporal"].mean(),
    "F1_Gain_vs_Old_GR": final_stable_comparison_df["F1_Gain_vs_Old_GR"].mean(),

    "ROC_Gain_vs_Static": final_stable_comparison_df["ROC_Gain_vs_Static"].mean(),
    "ROC_Gain_vs_Temporal": final_stable_comparison_df["ROC_Gain_vs_Temporal"].mean(),
    "ROC_Gain_vs_Old_GR": final_stable_comparison_df["ROC_Gain_vs_Old_GR"].mean()
}

final_stable_comparison_df = pd.concat(
    [final_stable_comparison_df, pd.DataFrame([avg_row])],
    ignore_index=True
)

display(final_stable_comparison_df)

final_stable_path = RESULTS_DIR / "FINAL_STABLE_RATE_FP_GR_COMPARISON_seed_42.csv"
final_stable_comparison_df.to_csv(final_stable_path, index=False)

print("Saved final stable RATE-FP-GR comparison to:")
print(final_stable_path)

,Project,Best_Static_F1,Best_Static_ROC_AUC,Best_Temporal_F1,Best_Temporal_ROC_AUC,Old_Graph_Risk_F1,Old_Graph_Risk_ROC_AUC,Stable_RATE_FP_GR_Model,Stable_RATE_FP_GR_F1,Stable_RATE_FP_GR_ROC_AUC,Stable_RATE_FP_GR_Accuracy,Stable_RATE_FP_GR_Precision,Stable_RATE_FP_GR_Recall,F1_Gain_vs_Static,F1_Gain_vs_Temporal,F1_Gain_vs_Old_GR,ROC_Gain_vs_Static,ROC_Gain_vs_Temporal,ROC_Gain_vs_Old_GR
0,jdt,0.686567,0.888588,0.704225,0.898346,0.705882,0.897804,RATE-FP-GR Extra Trees,0.774194,0.918135,0.906667,0.774194,0.774194,0.087626,0.069968,0.068311,0.029547,1.978856e-02,0.020331
1,pde,0.477612,0.803126,0.441176,0.808447,0.521739,0.811939,RATE-FP-GR Extra Trees,0.483871,0.808447,0.857778,0.483871,0.483871,0.006259,0.042694,-0.037868,0.005321,-1.110223e-16,-0.003492
2,equinox,0.682927,0.782456,0.651163,0.770175,0.585366,0.754386,RATE-FP-GR Balanced RF,0.526316,0.757018,0.632653,0.526316,0.526316,-0.156611,-0.124847,-0.059050,-0.025439,-1.315789e-02,0.002632
3,lucene,0.250000,0.700585,0.285714,0.705263,0.285714,0.748538,RATE-FP-GR Easy Ensemble,0.125000,0.718129,0.865385,0.142857,0.111111,-0.125000,-0.160714,-0.160714,0.017544,1.286550e-02,-0.030409
4,mylyn,0.536585,0.835836,0.495868,0.847181,0.369427,0.825270,RATE-FP-GR Balanced RF,0.510638,0.836614,0.835714,0.421053,0.648649,-0.025947,0.014771,0.141212,0.000779,-1.056612e-02,0.011345
5,Average,0.526738,0.802118,0.515629,0.805883,0.493626,0.807587,-,0.484004,0.807669,0.819639,0.469658,0.508828,-0.042735,-0.031626,-0.009622,0.005550,1.786008e-03,0.000081


Saved final stable RATE-FP-GR comparison to:
/content/drive/MyDrive/RATE_FP_Project/results/FINAL_STABLE_RATE_FP_GR_COMPARISON_seed_42.csv


In [ ]:
from sklearn.metrics import (
    average_precision_score,
    matthews_corrcoef,
    balanced_accuracy_score,
    precision_recall_curve,
    auc
)
import numpy as np
import pandas as pd


def ranking_metrics_at_k(y_true, y_prob, k_ratio=0.20):
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    n = len(y_true)
    k = max(1, int(np.ceil(k_ratio * n)))

    ranked_idx = np.argsort(y_prob)[::-1]
    top_k_idx = ranked_idx[:k]

    total_defects = y_true.sum()
    defects_found = y_true[top_k_idx].sum()

    recall_at_k = defects_found / total_defects if total_defects > 0 else 0
    precision_at_k = defects_found / k if k > 0 else 0

    base_defect_rate = total_defects / n if n > 0 else 0
    lift_at_k = precision_at_k / base_defect_rate if base_defect_rate > 0 else 0

    return {
        f"Recall@{int(k_ratio*100)}%": recall_at_k,
        f"Precision@{int(k_ratio*100)}%": precision_at_k,
        f"Lift@{int(k_ratio*100)}%": lift_at_k
    }


def extended_metrics(y_true, y_pred, y_prob):
    pr_auc = average_precision_score(y_true, y_prob)
    mcc = matthews_corrcoef(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)

    metrics = {
        "PR_AUC": pr_auc,
        "MCC": mcc,
        "Balanced_Accuracy": bal_acc
    }

    for k_ratio in [0.10, 0.20, 0.30]:
        metrics.update(ranking_metrics_at_k(y_true, y_prob, k_ratio))

    return metrics

In [ ]:
def risk_prioritization_experiment(project_name, rate_fp_df, random_state=42):
    print("\n" + "="*120)
    print(f"Risk-prioritization experiment: {project_name}")
    print("="*120)

    rate_fp_df = rate_fp_df.copy()

    for col in rate_fp_df.columns:
        if col != "classname":
            rate_fp_df[col] = pd.to_numeric(rate_fp_df[col], errors="coerce")

    rate_fp_df = rate_fp_df.fillna(0)
    rate_fp_df["is_defective"] = (rate_fp_df["bugs"] > 0).astype(int)

    target_cols = [
        "bugs", "nonTrivialBugs", "majorBugs",
        "criticalBugs", "highPriorityBugs", "is_defective"
    ]

    static_feature_cols = [
        col for col in rate_fp_df.columns
        if col not in ["classname"] + target_cols and "_temp_" not in col
    ]

    static_temporal_feature_cols = [
        col for col in rate_fp_df.columns
        if col not in ["classname"] + target_cols
    ]

    y = rate_fp_df["is_defective"].values
    indices_all = np.arange(len(rate_fp_df))

    train_idx, temp_idx, y_train, y_temp = train_test_split(
        indices_all,
        y,
        test_size=0.30,
        random_state=random_state,
        stratify=y
    )

    val_idx, test_idx, y_val, y_test = train_test_split(
        temp_idx,
        y_temp,
        test_size=0.50,
        random_state=random_state,
        stratify=y_temp
    )

    results = []

    # ======================================================
    # 1. Static baseline: RF + GBR + SVC + LR, select by validation PR-AUC
    # ======================================================
    X_static = rate_fp_df[static_feature_cols].values

    scaler_static = StandardScaler()
    X_static_train = scaler_static.fit_transform(X_static[train_idx])
    X_static_val = scaler_static.transform(X_static[val_idx])
    X_static_test = scaler_static.transform(X_static[test_idx])

    static_models = {
        "RF Static Risk": RandomForestClassifier(
            n_estimators=300,
            max_depth=10,
            class_weight="balanced",
            random_state=random_state
        ),
        "GBR Static Risk": GradientBoostingClassifier(
            n_estimators=300,
            max_depth=3,
            random_state=random_state
        ),
        "SVC Static Risk": SVC(
            kernel="rbf",
            C=10,
            probability=True,
            class_weight="balanced",
            random_state=random_state
        ),
        "LR Static Risk": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=random_state
        )
    }

    best_static_model = None
    best_static_score = -1
    best_static_info = None

    for name, model in static_models.items():
        model.fit(X_static_train, y[train_idx])

        val_prob = model.predict_proba(X_static_val)[:, 1]
        val_score = average_precision_score(y[val_idx], val_prob)

        if val_score > best_static_score:
            best_static_score = val_score
            best_static_model = model
            best_static_info = name

    static_test_prob = best_static_model.predict_proba(X_static_test)[:, 1]
    static_threshold, _ = find_best_threshold(y[val_idx], best_static_model.predict_proba(X_static_val)[:, 1])
    static_test_pred = predict_with_threshold(static_test_prob, static_threshold)

    static_metrics = evaluate_predictions(y[test_idx], static_test_pred, static_test_prob)
    static_metrics.update(extended_metrics(y[test_idx], static_test_pred, static_test_prob))
    static_metrics.update({
        "Project": project_name,
        "Model": best_static_info,
        "Family": "Static-only Risk Baseline"
    })
    results.append(static_metrics)

    # ======================================================
    # 2. Static + temporal baseline, select by validation PR-AUC
    # ======================================================
    X_st = rate_fp_df[static_temporal_feature_cols].values

    scaler_st = StandardScaler()
    X_train = scaler_st.fit_transform(X_st[train_idx])
    X_val = scaler_st.transform(X_st[val_idx])
    X_test = scaler_st.transform(X_st[test_idx])

    temporal_models = {
        "RF ST Risk": RandomForestClassifier(
            n_estimators=300,
            max_depth=10,
            class_weight="balanced",
            random_state=random_state
        ),
        "GBR ST Risk": GradientBoostingClassifier(
            n_estimators=300,
            max_depth=3,
            random_state=random_state
        ),
        "SVC ST Risk": SVC(
            kernel="rbf",
            C=10,
            probability=True,
            class_weight="balanced",
            random_state=random_state
        ),
        "LR ST Risk": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=random_state
        )
    }

    best_temporal_model = None
    best_temporal_score = -1
    best_temporal_info = None

    for name, model in temporal_models.items():
        model.fit(X_train, y[train_idx])

        val_prob = model.predict_proba(X_val)[:, 1]
        val_score = average_precision_score(y[val_idx], val_prob)

        if val_score > best_temporal_score:
            best_temporal_score = val_score
            best_temporal_model = model
            best_temporal_info = name

    temporal_val_prob = best_temporal_model.predict_proba(X_val)[:, 1]
    temporal_test_prob = best_temporal_model.predict_proba(X_test)[:, 1]
    temporal_threshold, _ = find_best_threshold(y[val_idx], temporal_val_prob)
    temporal_test_pred = predict_with_threshold(temporal_test_prob, temporal_threshold)

    temporal_metrics = evaluate_predictions(y[test_idx], temporal_test_pred, temporal_test_prob)
    temporal_metrics.update(extended_metrics(y[test_idx], temporal_test_pred, temporal_test_prob))
    temporal_metrics.update({
        "Project": project_name,
        "Model": best_temporal_info,
        "Family": "Static-Temporal Risk Baseline"
    })
    results.append(temporal_metrics)

    # ======================================================
    # 3. GraphSAGE graph-risk score
    # ======================================================
    X_graph_scaled = StandardScaler().fit_transform(X_st)

    x_tensor = torch.tensor(X_graph_scaled, dtype=torch.float)
    y_tensor = torch.tensor(y, dtype=torch.long)

    edge_index = build_similarity_graph(X_graph_scaled, k_neighbors=10)

    num_nodes = x_tensor.shape[0]

    train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(num_nodes, dtype=torch.bool)

    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True

    data_st = Data(
        x=x_tensor,
        edge_index=edge_index,
        y=y_tensor,
        train_mask=train_mask,
        val_mask=val_mask,
        test_mask=test_mask
    ).to(device)

    graph_model = GraphSAGEDefectPredictor(
        in_channels=data_st.x.shape[1],
        hidden_channels=64,
        out_channels=2,
        dropout=0.40
    ).to(device)

    class_counts = np.bincount(y[train_idx])
    class_weights = class_counts.sum() / (2.0 * class_counts)
    class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

    optimizer = torch.optim.AdamW(graph_model.parameters(), lr=0.003, weight_decay=1e-3)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_val_pr_auc = -1
    best_state = None
    patience = 50
    patience_counter = 0

    for epoch in range(1, 401):
        graph_model.train()
        optimizer.zero_grad()

        logits = graph_model(data_st.x, data_st.edge_index)
        loss = criterion(logits[data_st.train_mask], data_st.y[data_st.train_mask])

        loss.backward()
        optimizer.step()

        graph_model.eval()
        with torch.no_grad():
            val_logits = graph_model(data_st.x, data_st.edge_index)
            val_prob_all = F.softmax(val_logits, dim=1)[:, 1].detach().cpu().numpy()
            val_prob = val_prob_all[val_idx]

        val_pr_auc = average_precision_score(y[val_idx], val_prob)

        if val_pr_auc > best_val_pr_auc:
            best_val_pr_auc = val_pr_auc
            best_state = copy.deepcopy(graph_model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    if best_state is not None:
        graph_model.load_state_dict(best_state)

    graph_model.eval()
    with torch.no_grad():
        logits = graph_model(data_st.x, data_st.edge_index)
        graph_prob_all = F.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()

    graph_val_prob = graph_prob_all[val_idx]
    graph_test_prob = graph_prob_all[test_idx]
    graph_threshold, _ = find_best_threshold(y[val_idx], graph_val_prob)
    graph_test_pred = predict_with_threshold(graph_test_prob, graph_threshold)

    graph_metrics = evaluate_predictions(y[test_idx], graph_test_pred, graph_test_prob)
    graph_metrics.update(extended_metrics(y[test_idx], graph_test_pred, graph_test_prob))
    graph_metrics.update({
        "Project": project_name,
        "Model": "GraphSAGE Risk Score",
        "Family": "Graph-Risk Model"
    })
    results.append(graph_metrics)

    # ======================================================
    # 4. Static + temporal + graph-risk augmented ranking model
    # ======================================================
    graph_risk_feature = graph_prob_all.reshape(-1, 1)
    X_augmented = np.hstack([X_st, graph_risk_feature])

    scaler_aug = StandardScaler()
    X_aug_train = scaler_aug.fit_transform(X_augmented[train_idx])
    X_aug_val = scaler_aug.transform(X_augmented[val_idx])
    X_aug_test = scaler_aug.transform(X_augmented[test_idx])

    neg_count, pos_count = np.bincount(y[train_idx])
    scale_pos_weight = neg_count / max(pos_count, 1)

    gr_models = {
        "RATE-FP-GR ExtraTrees Risk": ExtraTreesClassifier(
            n_estimators=500,
            max_depth=12,
            class_weight="balanced",
            random_state=random_state
        ),
        "RATE-FP-GR BalancedRF Risk": BalancedRandomForestClassifier(
            n_estimators=500,
            max_depth=12,
            sampling_strategy="auto",
            replacement=True,
            random_state=random_state
        ),
        "RATE-FP-GR XGBoost Risk": XGBClassifier(
            n_estimators=500,
            max_depth=4,
            learning_rate=0.03,
            subsample=0.85,
            colsample_bytree=0.85,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=random_state
        )
    }

    best_gr_model = None
    best_gr_score = -1
    best_gr_info = None

    for name, model in gr_models.items():
        model.fit(X_aug_train, y[train_idx])

        val_prob = model.predict_proba(X_aug_val)[:, 1]
        val_score = average_precision_score(y[val_idx], val_prob)

        if val_score > best_gr_score:
            best_gr_score = val_score
            best_gr_model = model
            best_gr_info = name

    gr_val_prob = best_gr_model.predict_proba(X_aug_val)[:, 1]
    gr_test_prob = best_gr_model.predict_proba(X_aug_test)[:, 1]
    gr_threshold, _ = find_best_threshold(y[val_idx], gr_val_prob)
    gr_test_pred = predict_with_threshold(gr_test_prob, gr_threshold)

    gr_metrics = evaluate_predictions(y[test_idx], gr_test_pred, gr_test_prob)
    gr_metrics.update(extended_metrics(y[test_idx], gr_test_pred, gr_test_prob))
    gr_metrics.update({
        "Project": project_name,
        "Model": best_gr_info,
        "Family": "Proposed RATE-FP-GR Risk Prioritization"
    })
    results.append(gr_metrics)

    result_df = pd.DataFrame(results)

    print(f"Completed risk-prioritization {project_name}")
    print(result_df[["Model", "F1", "ROC_AUC", "PR_AUC", "Recall@20%", "Lift@20%"]])

    return result_df

In [ ]:
risk_results = []

for project_name, df in all_project_data.items():
    result_df = risk_prioritization_experiment(
        project_name,
        df,
        random_state=42
    )
    risk_results.append(result_df)

risk_all_df = pd.concat(risk_results, ignore_index=True)

risk_path = RESULTS_DIR / "RISK_PRIORITIZATION_RESULTS_seed_42.csv"
risk_all_df.to_csv(risk_path, index=False)

print("Saved risk-prioritization results to:")
print(risk_path)

display(risk_all_df)


Risk-prioritization experiment: jdt
Completed risk-prioritization jdt
                     Model        F1   ROC_AUC    PR_AUC  Recall@20%  Lift@20%
0          GBR Static Risk  0.686567  0.889943  0.709624    0.645161  3.225806
1               RF ST Risk  0.704225  0.898346  0.713426    0.612903  3.064516
2     GraphSAGE Risk Score  0.716418  0.895907  0.757960    0.709677  3.548387
3  RATE-FP-GR XGBoost Risk  0.690909  0.886690  0.756053    0.645161  3.225806

Risk-prioritization experiment: pde
Completed risk-prioritization pde
                        Model        F1   ROC_AUC    PR_AUC  Recall@20%  \
0              RF Static Risk  0.400000  0.806950  0.448142    0.580645   
1                  RF ST Risk  0.433735  0.807782  0.491381    0.548387   
2        GraphSAGE Risk Score  0.493151  0.807117  0.427062    0.580645   
3  RATE-FP-GR ExtraTrees Risk  0.433333  0.822414  0.472625    0.612903   

   Lift@20%  
0  2.903226  
1  2.741935  
2  2.903226  
3  3.064516  

Risk-prioritizat

,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,MCC,Balanced_Accuracy,Recall@10%,Precision@10%,Lift@10%,Recall@20%,Precision@20%,Lift@20%,Recall@30%,Precision@30%,Lift@30%,Project,Model,Family
0,0.860000,0.638889,0.741935,0.686567,0.889943,0.709624,0.599850,0.816346,0.387097,0.800000,3.870968,0.645161,0.666667,3.225806,0.806452,0.555556,2.688172,jdt,GBR Static Risk,Static-only Risk Baseline
1,0.860000,0.625000,0.806452,0.704225,0.898346,0.713426,0.623007,0.840201,0.387097,0.800000,3.870968,0.612903,0.633333,3.064516,0.838710,0.577778,2.795699,jdt,RF ST Risk,Static-Temporal Risk Baseline
2,0.873333,0.666667,0.774194,0.716418,0.895907,0.757960,0.638401,0.836677,0.419355,0.866667,4.193548,0.709677,0.733333,3.548387,0.806452,0.555556,2.688172,jdt,GraphSAGE Risk Score,Graph-Risk Model
3,0.886667,0.791667,0.612903,0.690909,0.886690,0.756053,0.630541,0.785443,0.419355,0.866667,4.193548,0.645161,0.666667,3.225806,0.806452,0.555556,2.688172,jdt,RATE-FP-GR XGBoost Risk,Proposed RATE-FP-GR Risk Prioritization
4,0.866667,0.526316,0.322581,0.400000,0.806950,0.448142,0.342356,0.638094,0.354839,0.478261,3.471248,0.580645,0.400000,2.903226,0.709677,0.323529,2.348197,pde,RF Static Risk,Static-only Risk Baseline
5,0.791111,0.346154,0.580645,0.433735,0.807782,0.491381,0.331457,0.702694,0.387097,0.521739,3.786816,0.548387,0.377778,2.741935,0.677419,0.308824,2.241461,pde,RF ST Risk,Static-Temporal Risk Baseline
6,0.835556,0.428571,0.580645,0.493151,0.807117,0.427062,0.404189,0.728467,0.387097,0.521739,3.786816,0.580645,0.400000,2.903226,0.677419,0.308824,2.241461,pde,GraphSAGE Risk Score,Graph-Risk Model
7,0.848889,0.448276,0.419355,0.433333,0.822414,0.472625,0.346522,0.668440,0.354839,0.478261,3.471248,0.612903,0.422222,3.064516,0.677419,0.308824,2.241461,pde,RATE-FP-GR ExtraTrees Risk,Proposed RATE-FP-GR Risk Prioritization
8,0.714286,0.600000,0.789474,0.681818,0.728070,0.564898,0.444591,0.728070,0.052632,0.200000,0.515789,0.263158,0.500000,1.289474,0.421053,0.533333,1.375439,equinox,SVC Static Risk,Static-only Risk Baseline
9,0.632653,0.523810,0.578947,0.550000,0.761404,0.695028,0.241825,0.622807,0.210526,0.800000,2.063158,0.315789,0.600000,1.547368,0.526316,0.666667,1.719298,equinox,RF ST Risk,Static-Temporal Risk Baseline


In [ ]:
risk_summary_rows = []

for project in risk_all_df["Project"].unique():
    project_df = risk_all_df[risk_all_df["Project"] == project].copy()

    static_row = project_df[project_df["Family"] == "Static-only Risk Baseline"].iloc[0]
    temporal_row = project_df[project_df["Family"] == "Static-Temporal Risk Baseline"].iloc[0]
    graph_row = project_df[project_df["Family"] == "Graph-Risk Model"].iloc[0]
    proposed_row = project_df[project_df["Family"] == "Proposed RATE-FP-GR Risk Prioritization"].iloc[0]

    risk_summary_rows.append({
        "Project": project,

        "Static_PR_AUC": static_row["PR_AUC"],
        "Temporal_PR_AUC": temporal_row["PR_AUC"],
        "Graph_PR_AUC": graph_row["PR_AUC"],
        "RATE_FP_GR_PR_AUC": proposed_row["PR_AUC"],

        "Static_Recall@20%": static_row["Recall@20%"],
        "Temporal_Recall@20%": temporal_row["Recall@20%"],
        "Graph_Recall@20%": graph_row["Recall@20%"],
        "RATE_FP_GR_Recall@20%": proposed_row["Recall@20%"],

        "Static_Lift@20%": static_row["Lift@20%"],
        "Temporal_Lift@20%": temporal_row["Lift@20%"],
        "Graph_Lift@20%": graph_row["Lift@20%"],
        "RATE_FP_GR_Lift@20%": proposed_row["Lift@20%"],

        "Static_F1": static_row["F1"],
        "Temporal_F1": temporal_row["F1"],
        "Graph_F1": graph_row["F1"],
        "RATE_FP_GR_F1": proposed_row["F1"],

        "PR_AUC_Gain_vs_Static": proposed_row["PR_AUC"] - static_row["PR_AUC"],
        "Recall20_Gain_vs_Static": proposed_row["Recall@20%"] - static_row["Recall@20%"],
        "Lift20_Gain_vs_Static": proposed_row["Lift@20%"] - static_row["Lift@20%"]
    })

risk_summary_df = pd.DataFrame(risk_summary_rows)

avg_row = {
    "Project": "Average",

    "Static_PR_AUC": risk_summary_df["Static_PR_AUC"].mean(),
    "Temporal_PR_AUC": risk_summary_df["Temporal_PR_AUC"].mean(),
    "Graph_PR_AUC": risk_summary_df["Graph_PR_AUC"].mean(),
    "RATE_FP_GR_PR_AUC": risk_summary_df["RATE_FP_GR_PR_AUC"].mean(),

    "Static_Recall@20%": risk_summary_df["Static_Recall@20%"].mean(),
    "Temporal_Recall@20%": risk_summary_df["Temporal_Recall@20%"].mean(),
    "Graph_Recall@20%": risk_summary_df["Graph_Recall@20%"].mean(),
    "RATE_FP_GR_Recall@20%": risk_summary_df["RATE_FP_GR_Recall@20%"].mean(),

    "Static_Lift@20%": risk_summary_df["Static_Lift@20%"].mean(),
    "Temporal_Lift@20%": risk_summary_df["Temporal_Lift@20%"].mean(),
    "Graph_Lift@20%": risk_summary_df["Graph_Lift@20%"].mean(),
    "RATE_FP_GR_Lift@20%": risk_summary_df["RATE_FP_GR_Lift@20%"].mean(),

    "Static_F1": risk_summary_df["Static_F1"].mean(),
    "Temporal_F1": risk_summary_df["Temporal_F1"].mean(),
    "Graph_F1": risk_summary_df["Graph_F1"].mean(),
    "RATE_FP_GR_F1": risk_summary_df["RATE_FP_GR_F1"].mean(),

    "PR_AUC_Gain_vs_Static": risk_summary_df["PR_AUC_Gain_vs_Static"].mean(),
    "Recall20_Gain_vs_Static": risk_summary_df["Recall20_Gain_vs_Static"].mean(),
    "Lift20_Gain_vs_Static": risk_summary_df["Lift20_Gain_vs_Static"].mean()
}

risk_summary_df = pd.concat(
    [risk_summary_df, pd.DataFrame([avg_row])],
    ignore_index=True
)

display(risk_summary_df)

risk_summary_path = RESULTS_DIR / "RISK_PRIORITIZATION_SUMMARY_seed_42.csv"
risk_summary_df.to_csv(risk_summary_path, index=False)

print("Saved risk-prioritization summary to:")
print(risk_summary_path)

,Project,Static_PR_AUC,Temporal_PR_AUC,Graph_PR_AUC,RATE_FP_GR_PR_AUC,Static_Recall@20%,Temporal_Recall@20%,Graph_Recall@20%,RATE_FP_GR_Recall@20%,Static_Lift@20%,Temporal_Lift@20%,Graph_Lift@20%,RATE_FP_GR_Lift@20%,Static_F1,Temporal_F1,Graph_F1,RATE_FP_GR_F1,PR_AUC_Gain_vs_Static,Recall20_Gain_vs_Static,Lift20_Gain_vs_Static
0,jdt,0.709624,0.713426,0.757960,0.756053,0.645161,0.612903,0.709677,0.645161,3.225806,3.064516,3.548387,3.225806,0.686567,0.704225,0.716418,0.690909,0.046428,0.000000,0.000000
1,pde,0.448142,0.491381,0.427062,0.472625,0.580645,0.548387,0.580645,0.612903,2.903226,2.741935,2.903226,3.064516,0.400000,0.433735,0.493151,0.433333,0.024483,0.032258,0.161290
2,equinox,0.564898,0.695028,0.714886,0.679107,0.263158,0.315789,0.368421,0.315789,1.289474,1.547368,1.805263,1.547368,0.681818,0.550000,0.564103,0.652174,0.114209,0.052632,0.257895
3,lucene,0.151862,0.212680,0.215287,0.211704,0.444444,0.333333,0.555556,0.333333,2.201058,1.650794,2.751323,1.650794,0.100000,0.125000,0.272727,0.270270,0.059843,-0.111111,-0.550265
4,mylyn,0.409681,0.444758,0.422820,0.416612,0.540541,0.648649,0.486486,0.648649,2.702703,3.243243,2.432432,3.243243,0.434783,0.495868,0.389831,0.472727,0.006932,0.108108,0.540541
5,Average,0.456841,0.511454,0.507603,0.507220,0.494790,0.491812,0.540157,0.511167,2.464453,2.449571,2.688126,2.546346,0.460634,0.461766,0.487246,0.503883,0.050379,0.016377,0.081892


Saved risk-prioritization summary to:
/content/drive/MyDrive/RATE_FP_Project/results/RISK_PRIORITIZATION_SUMMARY_seed_42.csv
